# Notebook 08 Main · Exploratory offline evaluation and blinded review

CPU. Attach expanded main Stages 05, 06 and 07 with the matching external manifests. Cells 1–12 build the expected opportunity ledger, collect execution evidence using the already qualified offline evaluator, and extract assertions. Cells 13–15 blind every extracted claim and export the full reviewer packet and the seeded Reviewer 2 sample. Reviewer 1 covers all claims. Reviewer 2 covers up to 300. Keep the main round’s blinding key private and restore the same key with `STAGE08_BLINDING_KEY_FILE` after a reset. Save the operator checkpoint ZIP privately, separately from the reviewer packet and the key. To resume a new session, set `STAGE08_RESUME_ARCHIVE` to the saved operator ZIP and `STAGE08_BLINDING_KEY_FILE` to the same private key. Rerun Cells 1–15 or 1–18 as appropriate. The protected checkpoint includes unblinded machine evidence and must stay with the operator.

Attach `reviewer_1_completed.csv` and `reviewer_2_completed.csv`, then run Cells 16–18. Send the adjudication packet only if the labels disagree. Attach `reviewer_3_completed.csv` if needed and run Cells 19–22. When the complete review passes, publish `stage_08_main_dataset` for Notebook 11.

This V2 notebook records every task identity through execution and freezes an expected seven-endpoint record frame for Notebook 11. Missing or duplicate task records block publication.


In [1]:
# Session 2 / 3 · NEW FIRST CELL (run before Cell 1): tell Notebook 08 where the key and checkpoint are
import os
from pathlib import Path

INPUT_ROOT = Path(os.environ.get("TEST_INPUT_ROOT", "/kaggle/input"))
CHECKPOINT_NAME = "stage_08_main_review_checkpoint.zip"   # session 3: stage_08_main_adjudication_checkpoint.zip

keys = [p for p in INPUT_ROOT.rglob("blinding_key.bin") if p.is_file()]
assert len(keys) == 1, f"expected exactly one blinding_key.bin under {INPUT_ROOT}, found {len(keys)}"
key = keys[0]
D = key.parent
zips = [p for p in D.glob(CHECKPOINT_NAME) if p.is_file()]
os.environ.pop("STAGE08_RESUME_ARCHIVE", None)
os.environ.pop("STAGE08_RESUME_ROOT", None)
os.environ["STAGE08_BLINDING_KEY_FILE"] = str(key)
if zips:
    os.environ["STAGE08_RESUME_ARCHIVE"] = str(zips[0])
    print("Resume from ZIP :", zips[0])
else:
    os.environ["STAGE08_RESUME_ROOT"] = str(D)
    print("Checkpoint zip was unpacked by Kaggle; resume from folder:", D)
print("Blinding key    :", key, "| bytes:", key.stat().st_size)
assert key.stat().st_size == 32, "blinding key must be 32 bytes"


Checkpoint zip was unpacked by Kaggle; resume from folder: /kaggle/input/datasets/anonuser02/granite-stage-08-ckpt-private
Blinding key    : /kaggle/input/datasets/anonuser02/granite-stage-08-ckpt-private/blinding_key.bin | bytes: 32


In [2]:
# ============================================================
# NOTEBOOK 08 — CELL 01
# Discover and qualify frozen Stage 05, 06 and 07 inputs.
# ============================================================

from __future__ import annotations

from collections import Counter, defaultdict
from pathlib import Path
import csv
import hashlib
import io
import json
import os
import time
import zipfile


STAGE08_VERSION = "stage08-main-v4-complete-accounting"
SOURCE_ROOT = Path(os.environ.get("STAGE08_SOURCE_ROOT", "/kaggle/input"))
PROJECT_ROOT = Path(os.environ.get(
    "STAGE08_PROJECT_ROOT", "/kaggle/working/prose_operation_experiment_stage08_main"
))
DATASET_ROOT = Path(os.environ.get(
    "STAGE08_DATASET_ROOT", "/kaggle/working/stage_08_main_dataset"
))
WORK_ROOT = Path(os.environ.get("STAGE08_MAIN_WORK_ROOT", "/kaggle/working/stage08_main_review"))
CHECKPOINT_ROOT = WORK_ROOT / "review_packets"
CHECKPOINT_ROOT.mkdir(parents=True, exist_ok=True)
PROJECT_ROOT.mkdir(parents=True, exist_ok=True)
DATASET_ROOT.mkdir(parents=True, exist_ok=True)


def sha256_bytes(data):
    return hashlib.sha256(data).hexdigest()


def sha256_file(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as stream:
        for chunk in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


def json_bytes(value):
    return (json.dumps(value, sort_keys=True, ensure_ascii=False,
                       separators=(",", ":")) + "\n").encode("utf-8")


def read_json(path):
    return json.loads(Path(path).read_text(encoding="utf-8"))


def read_jsonl(path):
    return [json.loads(line) for line in Path(path).read_text(encoding="utf-8").splitlines()
            if line.strip()]


def safe_relative(name):
    path = Path(name)
    if not name or path.is_absolute() or ".." in path.parts or str(path) != name:
        raise RuntimeError("Unsafe registered path: " + str(name))
    return path


def immutable_write(path, data):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    if path.is_file():
        if path.read_bytes() != data:
            raise RuntimeError("Immutable artifact changed: " + str(path))
        return "REUSED_IDENTICAL"
    if path.exists():
        raise RuntimeError("Non-file artifact at destination: " + str(path))
    temporary = path.with_name(path.name + ".tmp-" + str(os.getpid()))
    try:
        with temporary.open("xb") as stream:
            stream.write(data)
            stream.flush()
            os.fsync(stream.fileno())
        if path.is_file() and path.read_bytes() != data:
            raise RuntimeError("Concurrent immutable artifact conflict")
        os.replace(temporary, path)
    finally:
        temporary.unlink(missing_ok=True)
    return "WRITTEN"


def atomic_write(path, data):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_name(path.name + ".tmp-" + str(os.getpid()))
    try:
        with temporary.open("wb") as stream:
            stream.write(data)
            stream.flush()
            os.fsync(stream.fileno())
        os.replace(temporary, path)
    finally:
        temporary.unlink(missing_ok=True)


def immutable_json(path, value):
    return immutable_write(path, json_bytes(value))


def atomic_json(path, value):
    return atomic_write(path, json_bytes(value))


def write_jsonl(path, rows, immutable=True):
    payload = b"".join(json_bytes(row) for row in rows)
    return immutable_write(path, payload) if immutable else atomic_write(path, payload)


def write_csv(path, fields, records, immutable=True):
    buffer = io.StringIO(newline="")
    writer = csv.DictWriter(buffer, fieldnames=fields, lineterminator="\n",
                            extrasaction="raise")
    writer.writeheader()
    writer.writerows(records)
    payload = buffer.getvalue().encode("utf-8")
    return immutable_write(path, payload) if immutable else atomic_write(path, payload)


def report_cell(number, **data):
    report = {"notebook": "08", "cell": number, "status": "PASS",
              "version": STAGE08_VERSION, **data}
    atomic_json(PROJECT_ROOT / "reports/audit" / f"cell{number:02d}.json", report)
    print(json.dumps(report, indent=2))
    print(f"\nNOTEBOOK 08 — CELL {number:02d}: PASS")
    return report


def require_pass(number):
    path = PROJECT_ROOT / "reports/audit" / f"cell{number:02d}.json"
    if not path.is_file() or read_json(path).get("status") != "PASS":
        raise RuntimeError(f"Required Notebook 08 Cell {number:02d} has not passed")


# Resume is explicit and read-only. Kaggle may mount a checkpoint as a ZIP or
# expand it into several wrapper directories; both forms are supported.
resume_archive_name = os.environ.get("STAGE08_RESUME_ARCHIVE", "").strip()
resume_root_name = os.environ.get("STAGE08_RESUME_ROOT", "").strip()
if resume_archive_name and resume_root_name:
    raise RuntimeError("Configure only one Stage 08 resume input")
REVIEW_GATE_PATH = "reports/checkpoint_c_review_gate.json"


def verify_resume_version(gate):
    if gate.get("evaluation_version") != STAGE08_VERSION:
        raise RuntimeError("RESUME_BLOCK: checkpoint must be created by this Notebook 08 version")


if resume_archive_name:
    resume_archive = Path(resume_archive_name)
    if not resume_archive.is_file():
        raise RuntimeError("RESUME_BLOCK: checkpoint ZIP is missing")
    with zipfile.ZipFile(resume_archive) as archive:
        names = archive.namelist()
        if (len(names) != len(set(names)) or archive.testzip() is not None):
            raise RuntimeError("RESUME_BLOCK: checkpoint ZIP integrity failure")
        gate_name = "prose_operation_experiment/" + REVIEW_GATE_PATH
        if gate_name not in names:
            raise RuntimeError("RESUME_BLOCK: review gate missing")
        verify_resume_version(json.loads(archive.read(gate_name)))
        for name in names:
            if not name.startswith("prose_operation_experiment/") or name.endswith("/"):
                raise RuntimeError("RESUME_BLOCK: unexpected checkpoint member")
            relative = name[len("prose_operation_experiment/"):]
            if not relative or ".." in Path(relative).parts or Path(relative).is_absolute():
                raise RuntimeError("RESUME_BLOCK: unsafe checkpoint member")
            immutable_write(PROJECT_ROOT / relative, archive.read(name))
elif resume_root_name:
    previous = Path(resume_root_name)
    if not (previous / "reports/checkpoint_c_review_gate.json").is_file():
        matches = list(previous.rglob("reports/checkpoint_c_review_gate.json"))
        if len(matches) != 1:
            raise RuntimeError("RESUME_BLOCK: cannot identify unique Stage 08 checkpoint root")
        previous = matches[0].parent.parent
    verify_resume_version(read_json(previous / REVIEW_GATE_PATH))
    if previous.resolve() != PROJECT_ROOT.resolve():
        for source in sorted(previous.rglob("*")):
            if source.is_symlink():
                raise RuntimeError("RESUME_BLOCK: symlink in checkpoint")
            if source.is_file():
                immutable_write(PROJECT_ROOT / source.relative_to(previous),
                                source.read_bytes())


def discover_stage(stage):
    if not SOURCE_ROOT.is_dir():
        raise RuntimeError("Kaggle input mount is unavailable: " + str(SOURCE_ROOT))
    candidates = []
    for manifest_path in SOURCE_ROOT.rglob(f"registries/{stage}_manifest.json"):
        if manifest_path.is_symlink():
            continue
        project_root = manifest_path.parent.parent
        internal = read_json(manifest_path)
        if internal.get("stage") != stage or internal.get("status") != "PASS":
            continue
        for parent in project_root.parents:
            external_path = parent / f"stage_{stage}_main_manifest.json"
            if external_path.is_file():
                external = read_json(external_path)
                if (external.get("stage") == stage
                        and external.get("status") == "PASS"
                        and external.get("internal_manifest_sha256")
                        == sha256_file(manifest_path)):
                    candidates.append((project_root, manifest_path, external_path,
                                       internal, external))
                break
    if len(candidates) != 1:
        raise RuntimeError(f"Stage {stage}: expected one manifest-identified "
                           f"extracted project; found {len(candidates)}")
    root, internal_path, external_path, internal, external = candidates[0]
    return {"root": root, "internal_path": internal_path,
            "external_path": external_path, "internal": internal,
            "external": external}


STAGES = {stage: discover_stage(stage) for stage in ("05", "06", "07")}
SCOPE = 'main-seven-endpoint-exploratory-v1'
if any(release["internal"].get("protocol_scope") != SCOPE for release in STAGES.values()):
    raise RuntimeError("Attach the matching exploratory Stage 05, 06 and 07 packages.")
if any(info["internal"].get("design_id") != "openapi-fresh-pilot-v1" for info in STAGES.values()):
    raise RuntimeError("Attach fresh Stage 05, 06 and 07 releases.")
if STAGES["07"]["internal"]["upstream_manifest_sha256"]["06"] != sha256_file(STAGES["06"]["internal_path"]):
    raise RuntimeError("Stage 07 was not generated from the exact attached Stage 06 release.")

MAIN_OPERATIONS = int(STAGES["06"]["internal"]["operation_count"])
MAIN_TASKS = 2 * MAIN_OPERATIONS
if MAIN_OPERATIONS <= 0:
    raise RuntimeError("Empty main sample.")
if any(x["internal"].get("partition") != "main" for x in STAGES.values()):
    raise RuntimeError("Stage 08 requires main-mode inputs.")
if STAGES["06"]["internal"].get("task_count") != MAIN_TASKS:
    raise RuntimeError("Stage 06 task population differs from twice the frozen main operation count")
if STAGES["07"]["internal"].get("task_count") != MAIN_TASKS:
    raise RuntimeError("Stage 07 task population differs from twice the frozen main operation count")
if STAGES["07"]["internal"].get("operation_count") != MAIN_OPERATIONS:
    raise RuntimeError("Stage 07 operation population differs from the frozen main operation count")
SOURCE_IDENTITIES = {
    stage: {"internal_sha256": sha256_file(info["internal_path"]),
            "external_sha256": sha256_file(info["external_path"])}
    for stage, info in STAGES.items()
}
if STAGES["06"]["internal"].get("upstream_manifest_sha256", {}).get("05") != SOURCE_IDENTITIES["05"]["internal_sha256"]:
    raise RuntimeError("Stage 05 does not match the oracle release audited by Stage 06")
if resume_archive_name or resume_root_name:
    gate = read_json(PROJECT_ROOT / REVIEW_GATE_PATH)
    if gate.get("source_identities") != SOURCE_IDENTITIES:
        raise RuntimeError("RESUME_BLOCK: the frozen Stage 05–07 inputs differ")

report_cell(1, stage_roots={s: str(v["root"]) for s, v in STAGES.items()},
            upstream_manifest_identities="VERIFIED", expected_tasks=MAIN_TASKS,
            generation_performed_in_stage08=False)


{
  "notebook": "08",
  "cell": 1,
  "status": "PASS",
  "version": "stage08-main-v4-complete-accounting",
  "stage_roots": {
    "05": "/kaggle/input/datasets/anonuser02/stage-05-main/stage_05_main/prose_operation_experiment",
    "06": "/kaggle/input/datasets/anonuser02/granite-stage-07-input/stage_06_main_dataset/stage_06_main/stage_06_main/prose_operation_experiment",
    "07": "/kaggle/input/datasets/anonuser02/granite-stage-07-main/stage_07_main/stage_07_main/prose_operation_experiment"
  },
  "upstream_manifest_identities": "VERIFIED",
  "expected_tasks": 1700,
  "generation_performed_in_stage08": false
}

NOTEBOOK 08 — CELL 01: PASS


{'notebook': '08',
 'cell': 1,
 'status': 'PASS',
 'version': 'stage08-main-v4-complete-accounting',
 'stage_roots': {'05': '/kaggle/input/datasets/anonuser02/stage-05-main/stage_05_main/prose_operation_experiment',
  '06': '/kaggle/input/datasets/anonuser02/granite-stage-07-input/stage_06_main_dataset/stage_06_main/stage_06_main/prose_operation_experiment',
  '07': '/kaggle/input/datasets/anonuser02/granite-stage-07-main/stage_07_main/stage_07_main/prose_operation_experiment'},
 'upstream_manifest_identities': 'VERIFIED',
 'expected_tasks': 1700,
 'generation_performed_in_stage08': False}

In [3]:
# ============================================================
# NOTEBOOK 08 — CELL 02
# Verify frozen files, restore qualified evaluator and qualify CPU isolation.
# ============================================================

from __future__ import annotations

import csv
import importlib
import importlib.util
import json
import os
import platform
import sys
import time


import hashlib
import json
import re
from pathlib import PurePosixPath


ARTIFACT_SCHEMA = "openapi-artifact-v1"
PRODUCER_NOTEBOOKS = {
    "00": "00_protocol_and_runtime",
    "01": "01_corpus_preparation_and_eligibility",
    "02": "02_operation_slicing_and_prose_measurement",
    "03": "03_pair_and_evaluator_feasibility",
    "04": "04_strata_and_pilot_selection",
    "05": "05_oracle_build_and_review",
    "06": "06_build_corrected_evaluator_and_generation_tasks",
    "07": "07_pilot_generation",
    "08": "08_pilot_execution_and_review",
    "10": "10_power_and_main_sampling",
}


def provenance_digest(value):
    data = json.dumps(value, sort_keys=True, separators=(",", ":"),
                      ensure_ascii=False, allow_nan=False).encode("utf-8")
    return hashlib.sha256(data).hexdigest()


def artifact_path_for(record):
    paths = {record[key] for key in ("artifact_path", "relative_path", "path")
             if key in record}
    if len(paths) != 1:
        raise RuntimeError("Missing or inconsistent artifact paths")
    name = paths.pop()
    path = PurePosixPath(name)
    if not name or name == "." or path.is_absolute() or ".." in path.parts or "\\" in name or path.as_posix() != name:
        raise RuntimeError(f"Invalid artifact path: {name!r}")
    return name


def manifest_digest_valid(value):
    return isinstance(value, str) and re.fullmatch(r"[0-9a-f]{64}", value) is not None


def register_artifact_provenance(manifest):
    stage = manifest["stage"]
    artifacts = manifest["artifacts"]
    if "upstream_manifest_sha256" not in manifest:
        if stage != "00":
            raise RuntimeError(f"Stage {stage} has no upstream identity")
        manifest["upstream_manifest_sha256"] = {}
    links = manifest["upstream_manifest_sha256"]
    config_hash = manifest.get("configuration_hash")
    if config_hash is None:
        config_hash = provenance_digest({artifact_path_for(row): row["sha256"]
                                         for row in artifacts
                                         if artifact_path_for(row).startswith("config/")})
    manifest["configuration_hash"] = config_hash
    manifest["artifact_record_schema"] = ARTIFACT_SCHEMA
    upstream_hash = provenance_digest(links)
    for row in artifacts:
        row.update(artifact_path=artifact_path_for(row), schema_version=ARTIFACT_SCHEMA,
                   producer_notebook=PRODUCER_NOTEBOOKS[stage], config_hash=config_hash,
                   upstream_manifest_hash=upstream_hash)
    verify_artifact_provenance(manifest)


def verify_artifact_provenance(manifest):
    stage = manifest["stage"]
    if manifest.get("artifact_record_schema") != ARTIFACT_SCHEMA:
        raise RuntimeError(f"Stage {stage} requires a corrected provenance export")
    links = manifest["upstream_manifest_sha256"]
    config_hash = manifest["configuration_hash"]
    allowed = {key for key in PRODUCER_NOTEBOOKS if key < stage}
    if manifest.get("partition") == "main" and stage in ("05", "06"):
        allowed.add("10")
    if (not isinstance(links, dict) or bool(links) != (stage != "00")
            or any(key not in allowed or not manifest_digest_valid(value)
                   for key, value in links.items()) or not manifest_digest_valid(config_hash)):
        raise RuntimeError(f"Invalid Stage {stage} provenance")
    expected = (ARTIFACT_SCHEMA, PRODUCER_NOTEBOOKS[stage], config_hash, provenance_digest(links))
    seen = set()
    artifacts = manifest["artifacts"]
    if not artifacts or len(artifacts) != manifest["artifact_count"]:
        raise RuntimeError(f"Invalid Stage {stage} artifact count")
    for row in artifacts:
        name = artifact_path_for(row)
        actual = tuple(row.get(key) for key in
                       ("schema_version", "producer_notebook", "config_hash", "upstream_manifest_hash"))
        if (row.get("artifact_path") != name or name in seen or actual != expected
                or not manifest_digest_valid(row.get("sha256"))
                or type(row.get("bytes")) is not int or row["bytes"] < 0):
            raise RuntimeError(f"Invalid Stage {stage} artifact provenance: {name}")
        seen.add(name)


require_pass(1)
verified = {}
registered_by_stage = {}
for stage, info in STAGES.items():
    verify_artifact_provenance(info["internal"])
    artifacts = info["internal"]["artifacts"]
    if len(artifacts) != int(info["internal"]["artifact_count"]):
        raise RuntimeError("Upstream registered count mismatch: " + stage)
    seen = set()
    for artifact in artifacts:
        relative = artifact["relative_path"]
        safe_relative(relative)
        if relative in seen:
            raise RuntimeError("Duplicate registered input path: " + relative)
        seen.add(relative)
        path = info["root"] / relative
        if (not path.is_file() or path.is_symlink()
                or path.stat().st_size != int(artifact["bytes"])
                or sha256_file(path) != artifact["sha256"]):
            raise RuntimeError("Upstream integrity failure: " + stage + ":" + relative)
    verified[stage] = len(artifacts)
    registered_by_stage[stage] = seen

tasks_path = STAGES["06"]["root"] / "data/tasks.csv"
if sha256_file(tasks_path) != STAGES["07"]["internal"].get("task_ledger_sha256"):
    raise RuntimeError("Stage 07 generations do not match the corrected Stage 06 task ledger; rerun Notebook 07")
with tasks_path.open(newline="", encoding="utf-8") as stream:
    for task in csv.DictReader(stream):
        prompt = STAGES["06"]["root"] / safe_relative(task["prompt_path"])
        if sha256_file(prompt) != task["prompt_sha256"]:
            raise RuntimeError("Frozen task prompt differs: " + task["task_id"])

frozen_hashes = read_json(STAGES["06"]["root"] /
                          "registries/06_frozen_dependency_hashes.json")
for filename in ("src/worker.py", "src/routing.py", "src/metrics.py"):
    source = STAGES["06"]["root"] / filename
    if sha256_file(source) != frozen_hashes[filename]:
        raise RuntimeError("Qualified evaluator version changed: " + filename)
    immutable_write(PROJECT_ROOT / filename, source.read_bytes())

for source_stage, source_file, destination in (
    ("06", "config/metrics.yaml", "config/metrics.yaml"),
    ("06", "config/analysis_registered.yaml", "config/analysis_registered.yaml"),
    ("06", "data/tasks.csv", "data/tasks.csv"),
    ("07", "data/artifacts.csv", "data/artifacts.csv"),
):
    immutable_write(PROJECT_ROOT / destination,
                    (STAGES[source_stage]["root"] / source_file).read_bytes())

if str(PROJECT_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / "src"))
for module_name in ("worker", "routing", "metrics"):
    if module_name in sys.modules:
        loaded = Path(sys.modules[module_name].__file__).resolve()
        if loaded != (PROJECT_ROOT / "src" / f"{module_name}.py").resolve():
            raise RuntimeError("An unfrozen evaluator module is already imported: "
                               + module_name)
import routing
import worker
import metrics

if worker.WORKER_VERSION != "stage06-main-worker-v12-excluded-request-evidence":
    raise RuntimeError("Stage 06 qualified worker version differs")
if read_json(STAGES["06"]["root"] /
             "reports/qualification/isolation_preflight.json").get("status") != "PASS":
    raise RuntimeError("Stage 06 isolation qualification evidence is not PASS")

# No subprocess fallback. Kaggle must permit this real OS boundary.
ISOLATION = worker.qualify_os_boundary()
if ISOLATION.get("status") != "PASS":
    raise RuntimeError("SECURITY_BLOCK: Stage 06 OS boundary not available")

ENVIRONMENT = {"status": "PASS", "stage": "08", "os": platform.system(),
               "python": platform.python_version(), "worker_version": worker.WORKER_VERSION,
               "upstream_verified_artifacts": verified, "isolation": ISOLATION,
               "frozen_worker_sha256": frozen_hashes["src/worker.py"]}
immutable_json(PROJECT_ROOT / "reports/qualification/runtime_and_isolation.json",
               ENVIRONMENT)
report_cell(2, verified_artifacts=verified,
            sandbox_boundary="QUALIFIED_REAL_OS_BOUNDARY", gpu_required=False)


{
  "notebook": "08",
  "cell": 2,
  "status": "PASS",
  "version": "stage08-main-v4-complete-accounting",
  "verified_artifacts": {
    "05": 8511,
    "06": 2603,
    "07": 5046
  },
  "sandbox_boundary": "QUALIFIED_REAL_OS_BOUNDARY",
  "gpu_required": false
}

NOTEBOOK 08 — CELL 02: PASS


{'notebook': '08',
 'cell': 2,
 'status': 'PASS',
 'version': 'stage08-main-v4-complete-accounting',
 'verified_artifacts': {'05': 8511, '06': 2603, '07': 5046},
 'sandbox_boundary': 'QUALIFIED_REAL_OS_BOUNDARY',
 'gpu_required': False}

In [4]:
# ============================================================
# NOTEBOOK 08 — CELL 03
# Reconcile every frozen Main task, Stage 07 output and Stage 05 oracle package.
# ============================================================

from __future__ import annotations

from collections import Counter, defaultdict
import csv
import json


require_pass(2)
TASKS = list(csv.DictReader((PROJECT_ROOT / "data/tasks.csv").open(newline="")))
ARTIFACTS = list(csv.DictReader((PROJECT_ROOT / "data/artifacts.csv").open(newline="")))
if len(TASKS) != MAIN_TASKS or len(ARTIFACTS) != MAIN_TASKS:
    raise RuntimeError("Stage 08 requires one artifact row for every frozen Main task")
ARTIFACT_BY_TASK = {r["task_id"]: r for r in ARTIFACTS}
if len(ARTIFACT_BY_TASK) != MAIN_TASKS or set(ARTIFACT_BY_TASK) != {
    r["task_id"] for r in TASKS
}:
    raise RuntimeError("Stage 06 and Stage 07 task identities differ")

TASKS.sort(key=lambda r: int(r["order"]))
if any(r["partition"] != "main" for r in TASKS):
    raise RuntimeError("Stage 08 task ledger contains a non-main task.")
OP_IDS = {r["op_id"] for r in TASKS}
if len(OP_IDS) != MAIN_OPERATIONS:
    raise RuntimeError("Expected the frozen number of operation IDs")
PAIRS = defaultdict(set)
for task in TASKS:
    PAIRS[task["pair_id"]].add(task["condition"])
if len(PAIRS) != MAIN_OPERATIONS or any(conditions != {"L0", "L1"} for conditions in PAIRS.values()):
    raise RuntimeError("Expected the frozen main L0/L1 pairs")

excluded_audit = "reports/audit/excluded_scenarios_audit.jsonl"
if excluded_audit not in registered_by_stage["05"]:
    raise RuntimeError("The Stage 05 release lacks its frozen excluded scenario audit.")
EXCLUDED_BY_OP = defaultdict(list)
seen_excluded = set()
for entry in read_jsonl(STAGES["05"]["root"] / excluded_audit):
    op, scenario = entry["op_id"], entry["scenario"]
    key = (op, scenario["scenario_id"])
    if (op not in OP_IDS or scenario["op_id"] != op or key in seen_excluded
            or scenario["documented_response_mapping"] is not None
            or entry["reason"] != "NO_DOCUMENTED_RESPONSE_MAPPING"):
        raise RuntimeError("Invalid audit-only scenario identity")
    seen_excluded.add(key)
    EXCLUDED_BY_OP[op].append(scenario)

ORACLES = {}
for op_id in sorted(OP_IDS):
    folder = STAGES["05"]["root"] / "oracles/frozen/main" / op_id
    required = ("scenarios", "baselines", "legitimate_variations",
                "response_faults", "requirements")
    if not folder.is_dir():
        raise RuntimeError("Frozen Stage 05 oracle missing: " + op_id)
    records = {}
    for name in required:
        path = folder / f"{name}.jsonl"
        if (path.relative_to(STAGES["05"]["root"]).as_posix() not in registered_by_stage["05"]
                or not path.is_file() or path.is_symlink()):
            raise RuntimeError("Missing oracle record: " + str(path))
        records[name] = read_jsonl(path)
    if (folder / "response_fault_supplements.jsonl").exists():
        raise RuntimeError("Fresh oracle packages require one canonical response_faults.jsonl file.")
    document_relative = f"data/inputs/L0/{op_id}.json"
    document_path = STAGES["06"]["root"] / document_relative
    if (document_relative not in registered_by_stage["06"]
            or sha256_file(document_path) != frozen_hashes.get(document_relative)):
        raise RuntimeError("Operation document differs from Stage 06 qualification: " + op_id)
    records["operation_document"] = read_json(document_path)
    if not records["scenarios"] or not records["baselines"]:
        raise RuntimeError("No valid oracle replay data: " + op_id)
    if len({r["scenario_id"] for r in records["scenarios"]}) != len(records["scenarios"]):
        raise RuntimeError("Duplicate scenario identity: " + op_id)
    if len({r["baseline_id"] for r in records["baselines"]}) != len(records["baselines"]):
        raise RuntimeError("Duplicate baseline identity: " + op_id)
    ORACLES[op_id] = records

INPUT_INVENTORY = []
for task in TASKS:
    artifact = ARTIFACT_BY_TASK[task["task_id"]]
    raw_path = STAGES["07"]["root"] / task["expected_output_path"]
    if artifact["raw_hash"]:
        if not raw_path.is_file() or sha256_file(raw_path) != artifact["raw_hash"]:
            raise RuntimeError("Missing or modified generated raw output: " + task["task_id"])
        immutable_write(PROJECT_ROOT / task["expected_output_path"], raw_path.read_bytes())
    elif artifact["extraction_status"] != "no_output" or raw_path.exists() or artifact["extracted_hash"]:
        raise RuntimeError("Inconsistent no-output record: " + task["task_id"])
    if artifact["extracted_hash"]:
        relative = f"extracted/{artifact['artifact_id']}/test_generated.py"
        extracted = STAGES["07"]["root"] / relative
        if sha256_file(extracted) != artifact["extracted_hash"]:
            raise RuntimeError("Extracted code hash mismatch: " + task["task_id"])
        immutable_write(PROJECT_ROOT / relative, extracted.read_bytes())
    INPUT_INVENTORY.append({"task_id": task["task_id"], "pair_id": task["pair_id"],
                            "op_id": task["op_id"], "condition": task["condition"],
                            "artifact_id": artifact["artifact_id"] or None,
                            "raw_sha256": artifact["raw_hash"],
                            "extracted_sha256": artifact["extracted_hash"] or None,
                            "extraction_status": artifact["extraction_status"]})

write_jsonl(PROJECT_ROOT / "data/inventory/input_artifact_inventory.jsonl",
            INPUT_INVENTORY)
report_cell(3, tasks=MAIN_TASKS, operation_pairs=MAIN_OPERATIONS,
            raw_outputs=sum(bool(x["raw_sha256"]) for x in INPUT_INVENTORY), extracted_modules=sum(bool(x["extracted_sha256"])
                                                   for x in INPUT_INVENTORY),
            oracle_package_count=len(ORACLES))


{
  "notebook": "08",
  "cell": 3,
  "status": "PASS",
  "version": "stage08-main-v4-complete-accounting",
  "tasks": 1700,
  "operation_pairs": 850,
  "raw_outputs": 1700,
  "extracted_modules": 1616,
  "oracle_package_count": 850
}

NOTEBOOK 08 — CELL 03: PASS


{'notebook': '08',
 'cell': 3,
 'status': 'PASS',
 'version': 'stage08-main-v4-complete-accounting',
 'tasks': 1700,
 'operation_pairs': 850,
 'raw_outputs': 1700,
 'extracted_modules': 1616,
 'oracle_package_count': 850}

In [5]:
# ============================================================
# NOTEBOOK 08 — CELL 04
# Predeclare every baseline, scenario, variation and fault opportunity.
# ============================================================

from __future__ import annotations

from collections import Counter
import json


require_pass(3)
OPPORTUNITIES = []
for task in TASKS:
    records = ORACLES[task["op_id"]]
    seen = set()
    expected = (
        [("baseline", r["baseline_id"], r.get("scenario_id"))
         for r in records["baselines"]]
        + [("scenario", r["scenario_id"], r["scenario_id"])
           for r in records["scenarios"] if r.get("applicability") == "APPLICABLE"]
        + [("variation", r["variation_id"], None)
           for r in records["legitimate_variations"]]
        + [("fault", r["fault_id"], r.get("scenario_id"))
           for r in records["response_faults"] if r.get("applicable") is True]
    )
    for kind, opportunity_id, scenario_id in expected:
        key = (kind, opportunity_id)
        if key in seen:
            raise RuntimeError("Duplicate oracle opportunity: " + str(key))
        seen.add(key)
        OPPORTUNITIES.append({"opportunity_id": sha256_bytes(json_bytes(
            [task["task_id"], kind, opportunity_id]
        ))[:24], "task_id": task["task_id"], "op_id": task["op_id"],
            "kind": kind, "oracle_record_id": opportunity_id,
            "scenario_id": scenario_id,
            "replay_target_policy": (worker.REPLAY_TARGET_POLICY
                                     if kind in {"variation", "fault"} else None),
            "pre_execution_status": "EXPECTED"})
if len({r["opportunity_id"] for r in OPPORTUNITIES}) != len(OPPORTUNITIES):
    raise RuntimeError("Opportunity identity collision")
OPPORTUNITY_COUNTS = dict(Counter(row["kind"] for row in OPPORTUNITIES))
write_jsonl(PROJECT_ROOT / "data/evidence/opportunities_frozen.jsonl",
            OPPORTUNITIES)
report_cell(4, expected_tasks=MAIN_TASKS, opportunities=len(OPPORTUNITIES),
            opportunity_counts=OPPORTUNITY_COUNTS,
            frozen_before_first_execution=True)


{
  "notebook": "08",
  "cell": 4,
  "status": "PASS",
  "version": "stage08-main-v4-complete-accounting",
  "expected_tasks": 1700,
  "opportunities": 22238,
  "opportunity_counts": {
    "baseline": 1948,
    "scenario": 1948,
    "variation": 3700,
    "fault": 14642
  },
  "frozen_before_first_execution": true
}

NOTEBOOK 08 — CELL 04: PASS


{'notebook': '08',
 'cell': 4,
 'status': 'PASS',
 'version': 'stage08-main-v4-complete-accounting',
 'expected_tasks': 1700,
 'opportunities': 22238,
 'opportunity_counts': {'baseline': 1948,
  'scenario': 1948,
  'variation': 3700,
  'fault': 14642},
 'frozen_before_first_execution': True}

In [6]:
# ============================================================
# NOTEBOOK 08 — CELL 05
# Implement and qualify immutable-source claim decomposition and trace probes.
# ============================================================

# BEGIN_STAGE08_CLAIMS_MODULE

from __future__ import annotations

import ast
import base64
import hashlib
import json


CLAIMS_VERSION = "stage08-claims-v2-opcode-positions"
TRACE_MARKER = "STAGE08_REACHED_LINES_JSON="


def assertion_nodes(source):
    """Split conjunctions; keep OR, implications and other claims intact."""
    tree = ast.parse(source)
    result = []
    for node in ast.walk(tree):
        if not isinstance(node, ast.Assert):
            continue
        expressions = [node.test]
        while any(isinstance(x, ast.BoolOp) and isinstance(x.op, ast.And)
                  for x in expressions):
            next_items = []
            for expression in expressions:
                if isinstance(expression, ast.BoolOp) and isinstance(expression.op, ast.And):
                    next_items.extend(expression.values)
                else:
                    next_items.append(expression)
            expressions = next_items
        for index, expression in enumerate(expressions):
            result.append({
                "line": node.lineno,
                "expression_span": [expression.lineno, expression.col_offset,
                                    expression.end_lineno, expression.end_col_offset],
                "column": node.col_offset,
                "ordinal": index,
                "expression": ast.unparse(expression),
                "source_span": ast.get_source_segment(source, node) or "",
                "node_type": type(expression).__name__,
            })
    return sorted(result, key=lambda x: (x["line"], x["column"], x["ordinal"]))


def static_safety_warnings(source):
    tree = ast.parse(source)
    warnings = []
    banned_calls = {"eval", "exec", "__import__", "os.system", "os.popen",
                    "subprocess.run", "subprocess.Popen", "socket.socket",
                    "socket.create_connection"}
    for node in ast.walk(tree):
        if isinstance(node, ast.Call):
            name = ast.unparse(node.func)
            if name in banned_calls:
                warnings.append({"line": node.lineno, "risk": "dangerous_call", "name": name})
        if isinstance(node, ast.Import):
            for item in node.names:
                if item.name in ("ctypes", "subprocess", "socket"):
                    warnings.append({"line": node.lineno, "risk": "restricted_import",
                                     "name": item.name})
    return warnings


def observation_wrapper(source):
    """Trace executed expression positions in a supplemental isolated run."""
    encoded = base64.b64encode(source.encode("utf-8")).decode("ascii")
    lines = [
        "import atexit, base64, dis, json, sys",
        "_stage08_reached = {}",
        "_stage08_instructions = {}",
        "def _stage08_trace(frame, event, arg):",
        "    if frame.f_code.co_filename != '<frozen_generated_source>':",
        "        return None",
        "    if event == 'call':",
        "        frame.f_trace_opcodes = True",
        "        frame.f_trace_lines = False",
        "    elif event == 'opcode':",
        "        code = frame.f_code",
        "        if code not in _stage08_instructions:",
        "            _stage08_instructions[code] = {i.offset: i.positions for i in dis.get_instructions(code)}",
        "        position = _stage08_instructions[code].get(frame.f_lasti)",
        "        item = getattr(sys.modules['__main__'], 'current_item_id', None)",
        "        if item is not None and position is not None and None not in position:",
        "            _stage08_reached.setdefault(item, set()).add(tuple(position))",
        "    return _stage08_trace",
        "def _stage08_emit():",
        "    print('\\nSTAGE08_REACHED_LINES_JSON=' + json.dumps({item: sorted(spans) for item, spans in _stage08_reached.items()}))",
        "atexit.register(_stage08_emit)",
        "_stage08_entry = sys._getframe()",
        "_stage08_entry.f_trace_opcodes = True",
        "sys.settrace(_stage08_trace)",
        "exec(compile(base64.b64decode('" + encoded + "').decode('utf-8'),",
        "             '<frozen_generated_source>', 'exec', dont_inherit=True), globals())",
    ]
    return "\n".join(lines) + "\n"


def parse_observed_positions(stdout):
    values = [line[len(TRACE_MARKER):] for line in stdout.splitlines()
              if line.startswith(TRACE_MARKER)]
    if len(values) != 1:
        return None
    try:
        decoded = json.loads(values[0])
    except ValueError:
        return None
    if not isinstance(decoded, dict):
        return None
    for item, positions in decoded.items():
        if not item or not isinstance(positions, list):
            return None
        if any(not isinstance(span, list) or len(span) != 4
               or any(type(x) is not int or x < 0 for x in span)
               for span in positions):
            return None
    return decoded


def reached_items(node, observed):
    line, column, end_line, end_column = node["expression_span"]
    start, end = (line, column), (end_line, end_column)
    return sorted(item for item, positions in observed.items() if any(
        start <= (first_line, first_column) < (last_line, last_column) <= end
        for first_line, last_line, first_column, last_column in positions
    ))


# END_STAGE08_CLAIMS_MODULE

from pathlib import Path
import importlib.util
import json


require_pass(4)
shell = get_ipython()
if shell is None:
    raise RuntimeError("Notebook 08 source modules require a Jupyter kernel")
begin = "# BEGIN_STAGE08_" + "CLAIMS_MODULE"
end = "# END_STAGE08_" + "CLAIMS_MODULE"
matches = [raw for raw in reversed(shell.history_manager.input_hist_raw[-12:])
           if begin in raw and end in raw]
if not matches or matches[0].count(begin) != 1 or matches[0].count(end) != 1:
    raise RuntimeError("Cannot recover the visible, audited claims module source")
source = matches[0].split(begin, 1)[1].split(end, 1)[0].strip() + "\n"
module_path = PROJECT_ROOT / "src/claims.py"
immutable_write(module_path, source.encode("utf-8"))
spec = importlib.util.spec_from_file_location("stage08_claims", module_path)
claims = importlib.util.module_from_spec(spec)
spec.loader.exec_module(claims)
tests = claims.assertion_nodes("def test_a():\n    assert x and (y and z)\n")
if [x["expression"] for x in tests] != ["x", "y", "z"]:
    raise RuntimeError("Conjunctive assertion decomposition failed")
if claims.assertion_nodes("def f():\n    assert x or y\n")[0]["expression"] != "x or y":
    raise RuntimeError("Disjunctive claim preservation failed")
probe = claims.parse_observed_positions('STAGE08_REACHED_LINES_JSON={"test_a":[[2,2,11,12]]}')
if (not probe or claims.reached_items(tests[0], probe) != ["test_a"]
        or claims.reached_items(tests[1], probe)):
    raise RuntimeError("Short-circuit reachability fixture failed")
if not claims.static_safety_warnings("import subprocess\nsubprocess.run(['true'])\n"):
    raise RuntimeError("Safety check qualification failed")
if "<frozen_generated_source>" not in claims.observation_wrapper("assert True\n"):
    raise RuntimeError("Immutable-source trace wrapper qualification failed")
immutable_json(PROJECT_ROOT / "reports/qualification/claim_fixture_checks.json",
               {"status": "PASS", "fixtures": 5,
                "claims_module_sha256": sha256_file(module_path),
                "trace_is_supplemental_not_primary_execution": True})
report_cell(5, module_sha256=sha256_file(module_path), fixtures=5,
            generated_source_modification=False)


{
  "notebook": "08",
  "cell": 5,
  "status": "PASS",
  "version": "stage08-main-v4-complete-accounting",
  "module_sha256": "9912816f2f459c369b0dcc480e92c5918f729a90ba3d8e4341c66166e9f43c9e",
  "fixtures": 5,
  "generated_source_modification": false
}

NOTEBOOK 08 — CELL 05: PASS


{'notebook': '08',
 'cell': 5,
 'status': 'PASS',
 'version': 'stage08-main-v4-complete-accounting',
 'module_sha256': '9912816f2f459c369b0dcc480e92c5918f729a90ba3d8e4341c66166e9f43c9e',
 'fixtures': 5,
 'generated_source_modification': False}

In [7]:
# ============================================================
# NOTEBOOK 08 — CELL 06
# Separately audit extraction, compilation and static safety for 200 tasks.
# ============================================================

from __future__ import annotations

from collections import Counter
import ast
import json
import time


require_pass(5)
STATIC_RECORDS = []
for item in INPUT_INVENTORY:
    task_id = item["task_id"]
    entry = {"task_id": task_id,
             "extraction_status": item["extraction_status"],
             "compile_status": "NOT_APPLICABLE",
             "safety_status": "NOT_APPLICABLE", "diagnostic": None,
             "safety_warnings": [], "extracted_sha256": item["extracted_sha256"]}
    if item["extracted_sha256"]:
        code_path = PROJECT_ROOT / "extracted" / item["artifact_id"] / "test_generated.py"
        source = code_path.read_text(encoding="utf-8")
        try:
            compile(source, str(code_path), "exec")  # Never executes model code.
            entry["compile_status"] = "PASS"
        except (SyntaxError, UnicodeError, ValueError) as error:
            entry.update({"compile_status": "FAIL",
                          "diagnostic": type(error).__name__ + ": " + str(error)[:600]})
        if entry["compile_status"] == "PASS":
            warnings = claims.static_safety_warnings(source)
            entry["safety_warnings"] = warnings
            entry["safety_status"] = "DENIED" if warnings else "PASS"
    STATIC_RECORDS.append(entry)
write_jsonl(PROJECT_ROOT / "data/evidence/static_execution_ladder.jsonl",
            STATIC_RECORDS)
STATIC_BY_TASK = {row["task_id"]: row for row in STATIC_RECORDS}
report_cell(6, task_count=MAIN_TASKS,
            extraction_count=sum(bool(x["extracted_sha256"]) for x in STATIC_RECORDS),
            compilation=dict(Counter(x["compile_status"] for x in STATIC_RECORDS)),
            safety=dict(Counter(x["safety_status"] for x in STATIC_RECORDS)))


{
  "notebook": "08",
  "cell": 6,
  "status": "PASS",
  "version": "stage08-main-v4-complete-accounting",
  "task_count": 1700,
  "extraction_count": 1616,
  "compilation": {
    "PASS": 1584,
    "FAIL": 32,
    "NOT_APPLICABLE": 84
  },
  "safety": {
    "PASS": 1583,
    "NOT_APPLICABLE": 116,
    "DENIED": 1
  }
}

NOTEBOOK 08 — CELL 06: PASS


{'notebook': '08',
 'cell': 6,
 'status': 'PASS',
 'version': 'stage08-main-v4-complete-accounting',
 'task_count': 1700,
 'extraction_count': 1616,
 'compilation': {'PASS': 1584, 'FAIL': 32, 'NOT_APPLICABLE': 84},
 'safety': {'PASS': 1583, 'NOT_APPLICABLE': 116, 'DENIED': 1}}

In [8]:
# ============================================================
# NOTEBOOK 08 — CELL 07
# Collect pytest items only inside qualified Stage 06 OS isolation.
# ============================================================

from __future__ import annotations

from collections import Counter
import json
import re
import time


require_pass(6)


def frozen_route(op_id, mode="baseline", mutation=None):
    records = ORACLES[op_id]
    return {"scenarios": records["scenarios"], "baselines": records["baselines"],
            "operation_document": records["operation_document"],
            "excluded_scenarios": EXCLUDED_BY_OP[op_id], "mode": mode,
            "mutation": mutation}


def frozen_code(task_id):
    item = next(x for x in INPUT_INVENTORY if x["task_id"] == task_id)
    if not item["extracted_sha256"]:
        raise RuntimeError("No Python module for task: " + task_id)
    path = PROJECT_ROOT / "extracted" / item["artifact_id"] / "test_generated.py"
    if sha256_file(path) != item["extracted_sha256"]:
        raise RuntimeError("Frozen generated Python changed: " + task_id)
    return path.read_text(encoding="utf-8")


def isolated_result(task, route, phase="run", timeout=30):
    if task["task_id"] not in STATIC_BY_TASK:
        raise RuntimeError("Unknown task for isolated evaluator")
    # Always call the original qualified worker, never a local subprocess.
    result = worker.evaluate_isolated(
        frozen_code(task["task_id"]), route, phase=phase, timeout=timeout,
        maximum_output=200000)
    if result.get("status") == "EVALUATOR_FAILURE":
        raise RuntimeError(f"Evaluator failed at {phase} for {task['task_id']}: "
                           + result.get("stderr", "")[-1500:])
    if result.get("status") not in ("PASS", "TEST_FAILURE", "TIMEOUT", "OUTPUT_LIMIT",
                                    "EVALUATOR_UNAVAILABLE"):
        raise RuntimeError("Unexpected sandbox result classification")
    return result


COLLECTION_RECORDS = []
for index, task in enumerate(TASKS, 1):
    task_id = task["task_id"]
    static = STATIC_BY_TASK[task_id]
    destination = PROJECT_ROOT / "runs/main/collection" / (task_id + ".json")
    if destination.is_file():
        record = read_json(destination)
    elif static["compile_status"] != "PASS" or static["safety_status"] != "PASS":
        record = {"task_id": task_id, "status": "NOT_ELIGIBLE",
                  "item_ids": [], "diagnostic": "Compilation/safety/extraction not passed"}
        immutable_json(destination, record)
    else:
        result = isolated_result(task, frozen_route(task["op_id"]), phase="collect")
        collected = [line.strip() for line in result["stdout"].splitlines()
                     if re.search(r"(?:^|/)generated_test\.py::", line.strip())]
        record = {"task_id": task_id, "status": result["status"],
                  "item_ids": collected,
                  "has_skip_xfail_metadata": bool(re.search(
                      r"\b(?:skipped|xfailed|xpassed)\b", result["stdout"], re.I)),
                  "stderr": result["stderr"], "stdout": result["stdout"],
                  "request_attempts_during_collection": result["request_attempts"],
                  "duration_seconds": result["duration_seconds"]}
        immutable_json(destination, record)
    COLLECTION_RECORDS.append(record)
    if index % 20 == 0:
        print({"collection_checked": index, "expected": MAIN_TASKS})
write_jsonl(PROJECT_ROOT / "data/evidence/collection_results.jsonl",
            COLLECTION_RECORDS)
COLLECTION_BY_TASK = {row["task_id"]: row for row in COLLECTION_RECORDS}
report_cell(7, task_count=MAIN_TASKS,
            collection_status_counts=dict(Counter(x["status"] for x in COLLECTION_RECORDS)))


{'collection_checked': 20, 'expected': 1700}
{'collection_checked': 40, 'expected': 1700}
{'collection_checked': 60, 'expected': 1700}
{'collection_checked': 80, 'expected': 1700}
{'collection_checked': 100, 'expected': 1700}
{'collection_checked': 120, 'expected': 1700}
{'collection_checked': 140, 'expected': 1700}
{'collection_checked': 160, 'expected': 1700}
{'collection_checked': 180, 'expected': 1700}
{'collection_checked': 200, 'expected': 1700}
{'collection_checked': 220, 'expected': 1700}
{'collection_checked': 240, 'expected': 1700}
{'collection_checked': 260, 'expected': 1700}
{'collection_checked': 280, 'expected': 1700}
{'collection_checked': 300, 'expected': 1700}
{'collection_checked': 320, 'expected': 1700}
{'collection_checked': 340, 'expected': 1700}
{'collection_checked': 360, 'expected': 1700}
{'collection_checked': 380, 'expected': 1700}
{'collection_checked': 400, 'expected': 1700}
{'collection_checked': 420, 'expected': 1700}
{'collection_checked': 440, 'expected'

{'notebook': '08',
 'cell': 7,
 'status': 'PASS',
 'version': 'stage08-main-v4-complete-accounting',
 'task_count': 1700,
 'collection_status_counts': {'PASS': 873,
  'TEST_FAILURE': 710,
  'NOT_ELIGIBLE': 117}}

In [9]:
# ============================================================
# NOTEBOOK 08 — CELL 08
# Run fresh, isolated frozen baseline replays and preserve request evidence.
# ============================================================

from __future__ import annotations

from collections import Counter
import json
import time


require_pass(7)
BASELINE_RECORDS = []
REQUEST_ATTEMPTS = []
for index, task in enumerate(TASKS, 1):
    task_id = task["task_id"]
    collect = COLLECTION_BY_TASK[task_id]
    destination = PROJECT_ROOT / "runs/main/baseline" / (task_id + ".json")
    if destination.is_file():
        record = read_json(destination)
    elif collect["status"] != "PASS":
        unavailable = collect["status"] == "EVALUATOR_UNAVAILABLE"
        record = {"task_id": task_id,
                  "status": "EVALUATOR_UNAVAILABLE" if unavailable else "NOT_ELIGIBLE",
                  "baseline_pass": None if unavailable else False,
                  "request_attempts": collect.get("request_attempts_during_collection", []),
                  "item_outcomes": {},
                  "diagnostic": "Pytest collection was not successful"}
        immutable_json(destination, record)
    else:
        result = isolated_result(task, frozen_route(task["op_id"]), phase="run")
        if "SECURITY_BLOCK" in result.get("stderr", ""):
            raise RuntimeError("SECURITY_BLOCK: isolated baseline worker lost safety boundary")
        record = {"task_id": task_id, "status": result["status"],
                  "baseline_pass": (None if result["status"] == "EVALUATOR_UNAVAILABLE"
                                    else result["status"] == "PASS"),
                  "item_outcomes": result["item_outcomes"] or {},
                  "assertion_failures": result["assertion_failures"] or [],
                  "request_attempts": result["request_attempts"],
                  "stdout": result["stdout"], "stderr": result["stderr"],
                  "duration_seconds": result["duration_seconds"],
                  "collected_item_ids": collect["item_ids"]}
        immutable_json(destination, record)
    BASELINE_RECORDS.append(record)
    for position, attempt in enumerate(record["request_attempts"]):
        REQUEST_ATTEMPTS.append({
            "request_id": "reqattempt_" + sha256_bytes(json_bytes(
                [task_id, "baseline", position + 1]))[:24],
            "task_id": task_id, "artifact_id": ARTIFACT_BY_TASK[task_id]["artifact_id"],
            "op_id": task["op_id"],
            "pytest_item_ids": collect["item_ids"],
            "attempt_ordinal": position + 1,
            "phase": "baseline", **attempt})
    if index % 20 == 0:
        print({"baseline_checked": index, "expected": MAIN_TASKS})
write_jsonl(PROJECT_ROOT / "data/evidence/baseline_results.jsonl",
            BASELINE_RECORDS)
write_jsonl(PROJECT_ROOT / "data/evidence/request_attempts.jsonl",
            REQUEST_ATTEMPTS)
BASELINE_BY_TASK = {row["task_id"]: row for row in BASELINE_RECORDS}
report_cell(8, tasks=MAIN_TASKS,
            baseline_status_counts=dict(Counter(x["status"] for x in BASELINE_RECORDS)),
            recorded_request_attempts=len(REQUEST_ATTEMPTS))


{'baseline_checked': 20, 'expected': 1700}
{'baseline_checked': 40, 'expected': 1700}
{'baseline_checked': 60, 'expected': 1700}
{'baseline_checked': 80, 'expected': 1700}
{'baseline_checked': 100, 'expected': 1700}
{'baseline_checked': 120, 'expected': 1700}
{'baseline_checked': 140, 'expected': 1700}


{'baseline_checked': 160, 'expected': 1700}
{'baseline_checked': 180, 'expected': 1700}
{'baseline_checked': 200, 'expected': 1700}
{'baseline_checked': 220, 'expected': 1700}
{'baseline_checked': 240, 'expected': 1700}
{'baseline_checked': 260, 'expected': 1700}
{'baseline_checked': 280, 'expected': 1700}
{'baseline_checked': 300, 'expected': 1700}
{'baseline_checked': 320, 'expected': 1700}
{'baseline_checked': 340, 'expected': 1700}
{'baseline_checked': 360, 'expected': 1700}
{'baseline_checked': 380, 'expected': 1700}
{'baseline_checked': 400, 'expected': 1700}
{'baseline_checked': 420, 'expected': 1700}
{'baseline_checked': 440, 'expected': 1700}
{'baseline_checked': 460, 'expected': 1700}
{'baseline_checked': 480, 'expected': 1700}
{'baseline_checked': 500, 'expected': 1700}
{'baseline_checked': 520, 'expected': 1700}
{'baseline_checked': 540, 'expected': 1700}
{'baseline_checked': 560, 'expected': 1700}
{'baseline_checked': 580, 'expected': 1700}
{'baseline_checked': 600, 'expec

{
  "notebook": "08",
  "cell": 8,
  "status": "PASS",
  "version": "stage08-main-v4-complete-accounting",
  "tasks": 1700,
  "baseline_status_counts": {
    "TEST_FAILURE": 631,
    "NOT_ELIGIBLE": 827,
    "PASS": 84,
    "TIMEOUT": 113,
    "EVALUATOR_UNAVAILABLE": 45
  },
  "recorded_request_attempts": 401
}

NOTEBOOK 08 — CELL 08: PASS


{'notebook': '08',
 'cell': 8,
 'status': 'PASS',
 'version': 'stage08-main-v4-complete-accounting',
 'tasks': 1700,
 'baseline_status_counts': {'TEST_FAILURE': 631,
  'NOT_ELIGIBLE': 827,
  'PASS': 84,
  'TIMEOUT': 113,
  'EVALUATOR_UNAVAILABLE': 45},
 'recorded_request_attempts': 401}

In [10]:
# ============================================================
# NOTEBOOK 08 — CELL 09
# Independently replay legitimate variations against frozen route overlays.
# ============================================================

from __future__ import annotations

from collections import Counter
import json
import re
import time


require_pass(8)


def replay_evidence(task, mutation, mode):
    identifier = f"{mode}_id"
    task_id = task["task_id"]
    baseline_id = mutation["baseline_id"]
    baseline = BASELINE_BY_TASK[task_id]
    evidence = {"consumed": False, "target_baseline_id": baseline_id,
                "replay_target_policy": worker.REPLAY_TARGET_POLICY}
    if baseline["baseline_pass"] is None:
        return {**evidence, "status": "EVALUATOR_UNAVAILABLE",
                "diagnostic": "Frozen baseline could not be evaluated"}
    if not baseline["baseline_pass"]:
        return {**evidence, "status": "UNAVAILABLE_BASELINE",
                "diagnostic": "No passing frozen baseline"}
    route = frozen_route(task["op_id"], mode, mutation)
    scenario_id = next(row["scenario_id"] for row in route["baselines"]
                       if row["baseline_id"] == baseline_id)
    evidence["target_scenario_id"] = scenario_id

    # The chronological baseline log fixes the target before the replay starts.
    for index, attempt in enumerate(baseline["request_attempts"], 1):
        item = attempt.get("pytest_item_id")
        if (attempt["status"] == "DELIVERED"
                and attempt.get("baseline_id") == baseline_id
                and baseline["item_outcomes"].get(item) == "passed"):
            target = {key: attempt[key] for key in (
                "baseline_id", "scenario_id", "pytest_item_id",
                "item_request_ordinal", "request",
            )}
            evidence["target_baseline_request_id"] = "reqattempt_" + sha256_bytes(
                json_bytes([task_id, "baseline", index])
            )[:24]
            evidence["replay_target"] = target
            break
    else:
        return {**evidence, "status": "NOT_CONSUMED",
                "diagnostic": "No recorded target response in a passing baseline item"}

    route["replay_target"] = target
    result = isolated_result(task, route, phase="run")
    injections = [(index, row)
                  for index, row in enumerate(result["request_attempts"], 1)
                  if row.get("replay_mutation_applied") or row.get("fault_id")
                  or row.get("variation_id")]
    if len(injections) > 1:
        raise RuntimeError("Evaluator injected more than one response mutation")
    if injections:
        index, row = injections[0]
        if (row["status"] != "DELIVERED" or row.get("replay_mutation_applied") is not True
                or row.get(identifier) != mutation[identifier]
                or not worker.matches_replay_target(row, target)):
            raise RuntimeError("Evaluator injected a mutation outside the frozen request opportunity")
        evidence["replay_attempt_ordinal"] = index
    eligible = {target["pytest_item_id"]} if injections else set()
    worker_status = result["status"]
    failures = (eligible & set(result["assertion_failures"] or [])
                if worker_status == "TEST_FAILURE" else set())
    if worker_status == "EVALUATOR_UNAVAILABLE":
        status = "EVALUATOR_UNAVAILABLE"
    elif not injections:
        status = "NOT_CONSUMED"
        evidence["diagnostic"] = "The replay did not deliver a mutation at the recorded request"
    elif failures:
        status = "REJECTED_ASSERTION" if mode == "variation" else "KILLED_ASSERTION"
    elif worker_status == "PASS" and result["item_outcomes"].get(target["pytest_item_id"]) == "passed":
        status = "ACCEPTED" if mode == "variation" else "SURVIVED"
    else:
        status = "UNATTRIBUTED_OR_RUNTIME_FAILURE"
    return {**evidence, "status": status, "consumed": bool(injections),
            "worker_status": worker_status,
            "request_attempts": result["request_attempts"], "eligible_consumer_ids": sorted(eligible),
            "attributed_assertion_item_ids": sorted(failures), "item_outcomes": result["item_outcomes"],
            "stdout": result["stdout"], "stderr": result["stderr"],
            "duration_seconds": result["duration_seconds"]}


VARIATION_RESULTS = []
variation_opportunities = [r for r in OPPORTUNITIES if r["kind"] == "variation"]
task_lookup = {task["task_id"]: task for task in TASKS}
for index, opportunity in enumerate(variation_opportunities, 1):
    task = task_lookup[opportunity["task_id"]]
    variation = next(v for v in ORACLES[task["op_id"]]["legitimate_variations"]
                     if v["variation_id"] == opportunity["oracle_record_id"])
    destination = PROJECT_ROOT / "runs/main/variation" / (
        opportunity["opportunity_id"] + ".json")
    if destination.is_file():
        record = read_json(destination)
    else:
        evidence = replay_evidence(task, variation, "variation")
        accepted = {"ACCEPTED": True, "REJECTED_ASSERTION": False}.get(evidence["status"])
        record = {"opportunity_id": opportunity["opportunity_id"],
                  "task_id": task["task_id"], "variation_id": variation["variation_id"],
                  "accepted": accepted, **evidence}
        immutable_json(destination, record)
    VARIATION_RESULTS.append(record)
    if index % 50 == 0:
        print({"variations_reconciled": index, "total": len(variation_opportunities)})
write_jsonl(PROJECT_ROOT / "data/evidence/variation_results.jsonl", VARIATION_RESULTS)
report_cell(9, opportunity_count=len(variation_opportunities),
            status_counts=dict(Counter(x["status"] for x in VARIATION_RESULTS)))


{'variations_reconciled': 50, 'total': 3700}
{'variations_reconciled': 100, 'total': 3700}
{'variations_reconciled': 150, 'total': 3700}
{'variations_reconciled': 200, 'total': 3700}
{'variations_reconciled': 250, 'total': 3700}
{'variations_reconciled': 300, 'total': 3700}
{'variations_reconciled': 350, 'total': 3700}
{'variations_reconciled': 400, 'total': 3700}
{'variations_reconciled': 450, 'total': 3700}
{'variations_reconciled': 500, 'total': 3700}
{'variations_reconciled': 550, 'total': 3700}
{'variations_reconciled': 600, 'total': 3700}
{'variations_reconciled': 650, 'total': 3700}
{'variations_reconciled': 700, 'total': 3700}
{'variations_reconciled': 750, 'total': 3700}
{'variations_reconciled': 800, 'total': 3700}
{'variations_reconciled': 850, 'total': 3700}
{'variations_reconciled': 900, 'total': 3700}
{'variations_reconciled': 950, 'total': 3700}
{'variations_reconciled': 1000, 'total': 3700}
{'variations_reconciled': 1050, 'total': 3700}
{'variations_reconciled': 1100, '

{'variations_reconciled': 2150, 'total': 3700}
{'variations_reconciled': 2200, 'total': 3700}
{'variations_reconciled': 2250, 'total': 3700}
{'variations_reconciled': 2300, 'total': 3700}
{'variations_reconciled': 2350, 'total': 3700}
{'variations_reconciled': 2400, 'total': 3700}
{'variations_reconciled': 2450, 'total': 3700}
{'variations_reconciled': 2500, 'total': 3700}
{'variations_reconciled': 2550, 'total': 3700}
{'variations_reconciled': 2600, 'total': 3700}


{'variations_reconciled': 2650, 'total': 3700}
{'variations_reconciled': 2700, 'total': 3700}
{'variations_reconciled': 2750, 'total': 3700}
{'variations_reconciled': 2800, 'total': 3700}
{'variations_reconciled': 2850, 'total': 3700}
{'variations_reconciled': 2900, 'total': 3700}
{'variations_reconciled': 2950, 'total': 3700}
{'variations_reconciled': 3000, 'total': 3700}
{'variations_reconciled': 3050, 'total': 3700}
{'variations_reconciled': 3100, 'total': 3700}
{'variations_reconciled': 3150, 'total': 3700}
{'variations_reconciled': 3200, 'total': 3700}
{'variations_reconciled': 3250, 'total': 3700}
{'variations_reconciled': 3300, 'total': 3700}
{'variations_reconciled': 3350, 'total': 3700}
{'variations_reconciled': 3400, 'total': 3700}
{'variations_reconciled': 3450, 'total': 3700}
{'variations_reconciled': 3500, 'total': 3700}
{'variations_reconciled': 3550, 'total': 3700}
{'variations_reconciled': 3600, 'total': 3700}
{'variations_reconciled': 3650, 'total': 3700}
{'variations_

{'notebook': '08',
 'cell': 9,
 'status': 'PASS',
 'version': 'stage08-main-v4-complete-accounting',
 'opportunity_count': 3700,
 'status_counts': {'UNAVAILABLE_BASELINE': 3202,
  'REJECTED_ASSERTION': 4,
  'NOT_CONSUMED': 48,
  'ACCEPTED': 298,
  'EVALUATOR_UNAVAILABLE': 147,
  'UNATTRIBUTED_OR_RUNTIME_FAILURE': 1}}

In [11]:
# ============================================================
# NOTEBOOK 08 — CELL 10
# Replay each applicable response fault; require consumption and attribution.
# ============================================================

from __future__ import annotations

from collections import Counter
import json
import time


require_pass(9)
MUTATION_RESULTS = []
fault_opportunities = [r for r in OPPORTUNITIES if r["kind"] == "fault"]
for index, opportunity in enumerate(fault_opportunities, 1):
    task = task_lookup[opportunity["task_id"]]
    fault = next(f for f in ORACLES[task["op_id"]]["response_faults"]
                 if f["fault_id"] == opportunity["oracle_record_id"])
    destination = PROJECT_ROOT / "runs/main/fault" / (
        opportunity["opportunity_id"] + ".json")
    if destination.is_file():
        record = read_json(destination)
    else:
        evidence = replay_evidence(task, fault, "fault")
        killed = {"KILLED_ASSERTION": True, "SURVIVED": False}.get(evidence["status"])
        record = {"opportunity_id": opportunity["opportunity_id"],
                  "task_id": task["task_id"], "fault_id": fault["fault_id"],
                  "killed": killed, "failure_attribution": "assertion" if killed else "",
                  **evidence}
        immutable_json(destination, record)
    MUTATION_RESULTS.append(record)
    if index % 100 == 0:
        print({"faults_reconciled": index, "total": len(fault_opportunities)})
write_jsonl(PROJECT_ROOT / "data/evidence/mutation_results.jsonl", MUTATION_RESULTS)
report_cell(10, opportunity_count=len(fault_opportunities),
            status_counts=dict(Counter(x["status"] for x in MUTATION_RESULTS)))


{'faults_reconciled': 100, 'total': 14642}
{'faults_reconciled': 200, 'total': 14642}
{'faults_reconciled': 300, 'total': 14642}


{'faults_reconciled': 400, 'total': 14642}
{'faults_reconciled': 500, 'total': 14642}
{'faults_reconciled': 600, 'total': 14642}
{'faults_reconciled': 700, 'total': 14642}
{'faults_reconciled': 800, 'total': 14642}
{'faults_reconciled': 900, 'total': 14642}
{'faults_reconciled': 1000, 'total': 14642}
{'faults_reconciled': 1100, 'total': 14642}
{'faults_reconciled': 1200, 'total': 14642}
{'faults_reconciled': 1300, 'total': 14642}
{'faults_reconciled': 1400, 'total': 14642}
{'faults_reconciled': 1500, 'total': 14642}
{'faults_reconciled': 1600, 'total': 14642}
{'faults_reconciled': 1700, 'total': 14642}
{'faults_reconciled': 1800, 'total': 14642}
{'faults_reconciled': 1900, 'total': 14642}
{'faults_reconciled': 2000, 'total': 14642}
{'faults_reconciled': 2100, 'total': 14642}
{'faults_reconciled': 2200, 'total': 14642}
{'faults_reconciled': 2300, 'total': 14642}
{'faults_reconciled': 2400, 'total': 14642}
{'faults_reconciled': 2500, 'total': 14642}
{'faults_reconciled': 2600, 'total': 1

{'faults_reconciled': 2800, 'total': 14642}
{'faults_reconciled': 2900, 'total': 14642}
{'faults_reconciled': 3000, 'total': 14642}


{'faults_reconciled': 3100, 'total': 14642}
{'faults_reconciled': 3200, 'total': 14642}
{'faults_reconciled': 3300, 'total': 14642}
{'faults_reconciled': 3400, 'total': 14642}
{'faults_reconciled': 3500, 'total': 14642}
{'faults_reconciled': 3600, 'total': 14642}
{'faults_reconciled': 3700, 'total': 14642}
{'faults_reconciled': 3800, 'total': 14642}
{'faults_reconciled': 3900, 'total': 14642}
{'faults_reconciled': 4000, 'total': 14642}
{'faults_reconciled': 4100, 'total': 14642}
{'faults_reconciled': 4200, 'total': 14642}
{'faults_reconciled': 4300, 'total': 14642}
{'faults_reconciled': 4400, 'total': 14642}
{'faults_reconciled': 4500, 'total': 14642}
{'faults_reconciled': 4600, 'total': 14642}
{'faults_reconciled': 4700, 'total': 14642}
{'faults_reconciled': 4800, 'total': 14642}
{'faults_reconciled': 4900, 'total': 14642}
{'faults_reconciled': 5000, 'total': 14642}
{'faults_reconciled': 5100, 'total': 14642}
{'faults_reconciled': 5200, 'total': 14642}
{'faults_reconciled': 5300, 'tot

{'faults_reconciled': 5400, 'total': 14642}
{'faults_reconciled': 5500, 'total': 14642}
{'faults_reconciled': 5600, 'total': 14642}


{'faults_reconciled': 5700, 'total': 14642}
{'faults_reconciled': 5800, 'total': 14642}
{'faults_reconciled': 5900, 'total': 14642}
{'faults_reconciled': 6000, 'total': 14642}
{'faults_reconciled': 6100, 'total': 14642}
{'faults_reconciled': 6200, 'total': 14642}
{'faults_reconciled': 6300, 'total': 14642}
{'faults_reconciled': 6400, 'total': 14642}
{'faults_reconciled': 6500, 'total': 14642}
{'faults_reconciled': 6600, 'total': 14642}
{'faults_reconciled': 6700, 'total': 14642}
{'faults_reconciled': 6800, 'total': 14642}
{'faults_reconciled': 6900, 'total': 14642}
{'faults_reconciled': 7000, 'total': 14642}
{'faults_reconciled': 7100, 'total': 14642}
{'faults_reconciled': 7200, 'total': 14642}
{'faults_reconciled': 7300, 'total': 14642}
{'faults_reconciled': 7400, 'total': 14642}
{'faults_reconciled': 7500, 'total': 14642}
{'faults_reconciled': 7600, 'total': 14642}
{'faults_reconciled': 7700, 'total': 14642}
{'faults_reconciled': 7800, 'total': 14642}
{'faults_reconciled': 7900, 'tot

... (4 outputs left out)

{'faults_reconciled': 13200, 'total': 14642}
{'faults_reconciled': 13300, 'total': 14642}
{'faults_reconciled': 13400, 'total': 14642}


{'faults_reconciled': 13500, 'total': 14642}
{'faults_reconciled': 13600, 'total': 14642}
{'faults_reconciled': 13700, 'total': 14642}
{'faults_reconciled': 13800, 'total': 14642}
{'faults_reconciled': 13900, 'total': 14642}
{'faults_reconciled': 14000, 'total': 14642}
{'faults_reconciled': 14100, 'total': 14642}
{'faults_reconciled': 14200, 'total': 14642}
{'faults_reconciled': 14300, 'total': 14642}
{'faults_reconciled': 14400, 'total': 14642}
{'faults_reconciled': 14500, 'total': 14642}
{'faults_reconciled': 14600, 'total': 14642}


{
  "notebook": "08",
  "cell": 10,
  "status": "PASS",
  "version": "stage08-main-v4-complete-accounting",
  "opportunity_count": 14642,
  "status_counts": {
    "UNAVAILABLE_BASELINE": 13185,
    "SURVIVED": 657,
    "NOT_CONSUMED": 181,
    "KILLED_ASSERTION": 93,
    "EVALUATOR_UNAVAILABLE": 516,
    "UNATTRIBUTED_OR_RUNTIME_FAILURE": 10
  }
}

NOTEBOOK 08 — CELL 10: PASS


{'notebook': '08',
 'cell': 10,
 'status': 'PASS',
 'version': 'stage08-main-v4-complete-accounting',
 'opportunity_count': 14642,
 'status_counts': {'UNAVAILABLE_BASELINE': 13185,
  'SURVIVED': 657,
  'NOT_CONSUMED': 181,
  'KILLED_ASSERTION': 93,
  'EVALUATOR_UNAVAILABLE': 516,
  'UNATTRIBUTED_OR_RUNTIME_FAILURE': 10}}

In [12]:
# ============================================================
# NOTEBOOK 08 — CELL 11
# Reconcile complete execution/opportunity evidence without hiding losses.
# ============================================================

from __future__ import annotations

from collections import Counter
import csv
import json
import time


require_pass(10)
var_lookup = {r["opportunity_id"]: r for r in VARIATION_RESULTS}
fault_lookup = {r["opportunity_id"]: r for r in MUTATION_RESULTS}
if len(var_lookup) != len(VARIATION_RESULTS) or len(fault_lookup) != len(MUTATION_RESULTS):
    raise RuntimeError("Duplicate finalized replay opportunity")

EXECUTION_ROWS = []
for task in TASKS:
    task_id = task["task_id"]
    artifact = ARTIFACT_BY_TASK[task_id]
    static = STATIC_BY_TASK[task_id]
    collection = COLLECTION_BY_TASK[task_id]
    baseline = BASELINE_BY_TASK[task_id]
    EXECUTION_ROWS.append({
        "task_id": task_id,
        "op_id": task["op_id"],
        "pair_id": task["pair_id"],
        "condition": task["condition"],
        "artifact_id": artifact["artifact_id"],
        "extraction_status": artifact["extraction_status"],
        "compile_status": static["compile_status"],
        "safety_status": static["safety_status"],
        "collection_status": collection["status"],
        "collected_item_count": len(collection["item_ids"]),
        "item_ids_json": json.dumps(collection["item_ids"], separators=(",", ":")),
        "baseline_status": baseline["status"],
        "baseline_pass": baseline["baseline_pass"],
        "collection_evaluator_unavailable": collection["status"] == "EVALUATOR_UNAVAILABLE",
        "baseline_evaluator_unavailable": baseline["status"] == "EVALUATOR_UNAVAILABLE",
        "no_request": not baseline["request_attempts"],
        "diagnostic": static["diagnostic"] or baseline.get("diagnostic") or "",
    })
EXECUTION_COLUMNS = list(EXECUTION_ROWS[0])
write_csv(PROJECT_ROOT / "data/evidence/execution.csv", EXECUTION_COLUMNS, EXECUTION_ROWS)

SCENARIO_HITS = []
for op in OPPORTUNITIES:
    if op["kind"] != "scenario":
        continue
    baseline = BASELINE_BY_TASK[op["task_id"]]
    hits = [i for i, attempt in enumerate(baseline["request_attempts"], 1)
            if (attempt.get("classification") or {}).get("classifiable") is True
            and attempt.get("scenario_id") == op["oracle_record_id"]]
    SCENARIO_HITS.append({"task_id": op["task_id"],
                          "artifact_id": ARTIFACT_BY_TASK[op["task_id"]]["artifact_id"],
                          "scenario_id": op["oracle_record_id"],
                          "observed": bool(hits), "first_hit": hits[0] if hits else "",
                          "dedup_key": op["oracle_record_id"],
                          "baseline_status": baseline["status"]})
write_csv(PROJECT_ROOT / "data/evidence/scenario_hits.csv", list(SCENARIO_HITS[0]),
          SCENARIO_HITS)

RECONCILED_OPPORTUNITIES = []
for op in OPPORTUNITIES:
    task_id = op["task_id"]
    baseline = BASELINE_BY_TASK[task_id]
    if op["kind"] == "variation":
        row = var_lookup[op["opportunity_id"]]
        status = row["status"]
    elif op["kind"] == "fault":
        row = fault_lookup[op["opportunity_id"]]
        status = row["status"]
    elif op["kind"] == "scenario":
        status = ("OBSERVED" if any(
            (req.get("classification") or {}).get("classifiable") is True
            and req.get("scenario_id") == op["oracle_record_id"]
            for req in baseline["request_attempts"]) else "NOT_OBSERVED")
    elif op["kind"] == "baseline":
        consumed = any(row.get("status") == "DELIVERED"
                       and row.get("baseline_id") == op["oracle_record_id"]
                       for row in baseline["request_attempts"])
        status = ("EVALUATOR_UNAVAILABLE" if baseline["baseline_pass"] is None else
                  baseline["status"] if consumed else
                  "NOT_CONSUMED" if COLLECTION_BY_TASK[task_id]["status"] == "PASS"
                  else baseline["status"])
    else:
        raise RuntimeError("Unregistered opportunity kind")
    RECONCILED_OPPORTUNITIES.append({**op, "final_status": status})

if len(RECONCILED_OPPORTUNITIES) != len(OPPORTUNITIES):
    raise RuntimeError("Opportunity population changed during replay")
write_jsonl(PROJECT_ROOT / "data/evidence/opportunity_outcomes.jsonl",
            RECONCILED_OPPORTUNITIES)

# Stable tabular exports are consumed by Notebook 09. Retain the richer
# per-opportunity JSONL records as well as these complete CSV summaries.
for kind, records, filename, identifiers in (
    ("variation", VARIATION_RESULTS, "variation_results.csv", "variation_id"),
    ("fault", MUTATION_RESULTS, "mutation_results.csv", "fault_id"),
):
    by_id = {r["opportunity_id"]: r for r in records}
    exported = []
    for opportunity in OPPORTUNITIES:
        if opportunity["kind"] != kind:
            continue
        outcome = by_id[opportunity["opportunity_id"]]
        oracle_rows = (ORACLES[opportunity["op_id"]]["legitimate_variations"]
                       if kind == "variation" else
                       ORACLES[opportunity["op_id"]]["response_faults"])
        original = next(r for r in oracle_rows
                        if r[identifiers] == opportunity["oracle_record_id"])
        exported.append({
            "opportunity_id": opportunity["opportunity_id"],
            "task_id": opportunity["task_id"],
            "artifact_id": ARTIFACT_BY_TASK[opportunity["task_id"]]["artifact_id"],
            "op_id": opportunity["op_id"],
            "baseline_id": original["baseline_id"],
            identifiers: opportunity["oracle_record_id"],
            "scheduled": bool(outcome.get("worker_status")),
            "status": outcome["status"],
            "consumed": outcome.get("consumed", False),
            "accepted": outcome.get("accepted", ""),
            "killed": outcome.get("killed", ""),
            "failure_attribution": outcome.get("failure_attribution", ""),
            "reason": outcome.get("reason", outcome.get("diagnostic", "")),
            "worker_status": outcome.get("worker_status", ""),
            "replay_target_policy": outcome["replay_target_policy"],
            "target_baseline_request_id": outcome.get("target_baseline_request_id", ""),
            "target_pytest_item_id": outcome.get("replay_target", {}).get("pytest_item_id", ""),
            "target_item_request_ordinal": outcome.get("replay_target", {}).get("item_request_ordinal", ""),
            "replay_attempt_ordinal": outcome.get("replay_attempt_ordinal", ""),
        })
    fields = ["opportunity_id", "task_id", "artifact_id", "op_id",
              "baseline_id", identifiers, "scheduled", "status", "consumed",
              "accepted", "killed", "failure_attribution", "reason", "worker_status",
              "replay_target_policy", "target_baseline_request_id", "target_pytest_item_id",
              "target_item_request_ordinal", "replay_attempt_ordinal"]
    write_csv(PROJECT_ROOT / "data/evidence" / filename, fields, exported)

ITEM_OUTCOMES = []
for task in TASKS:
    task_id = task["task_id"]
    collection = COLLECTION_BY_TASK[task_id]
    baseline = BASELINE_BY_TASK[task_id]
    for pytest_item_id in collection["item_ids"]:
        outcome = baseline["item_outcomes"].get(pytest_item_id, "UNAVAILABLE")
        ITEM_OUTCOMES.append({"task_id": task_id, "pytest_item_id": pytest_item_id,
                              "outcome": outcome,
                              "collection_status": collection["status"],
                              "baseline_status": baseline["status"]})
write_csv(PROJECT_ROOT / "data/evidence/pytest_item_outcomes.csv",
          ["task_id", "pytest_item_id", "outcome", "collection_status", "baseline_status"],
          ITEM_OUTCOMES)

ALL_REQUEST_ATTEMPTS = list(REQUEST_ATTEMPTS)
for kind, records in (("variation", VARIATION_RESULTS),
                      ("fault", MUTATION_RESULTS)):
    for record in records:
        for index, attempt in enumerate(record.get("request_attempts", []), 1):
            ALL_REQUEST_ATTEMPTS.append({
                                         "request_id": "reqattempt_" + sha256_bytes(json_bytes(
                                             [record["task_id"], kind,
                                              record["opportunity_id"], index]))[:24],
                                         "task_id": record["task_id"],
                                         "artifact_id": ARTIFACT_BY_TASK[record["task_id"]]["artifact_id"],
                                         "phase": kind,
                                         "opportunity_id": record["opportunity_id"],
                                         "attempt_ordinal": index, **attempt})
write_jsonl(PROJECT_ROOT / "data/evidence/request_attempts_all.jsonl",
            ALL_REQUEST_ATTEMPTS)
report_cell(11, tasks=len(EXECUTION_ROWS),
            opportunities=len(RECONCILED_OPPORTUNITIES),
            request_attempts=len(ALL_REQUEST_ATTEMPTS),
            collected_item_outcomes=len(ITEM_OUTCOMES),
            no_silent_population_exclusion=True)


{
  "notebook": "08",
  "cell": 11,
  "status": "PASS",
  "version": "stage08-main-v4-complete-accounting",
  "tasks": 1700,
  "opportunities": 22238,
  "request_attempts": 1897,
  "collected_item_outcomes": 1012,
  "no_silent_population_exclusion": true
}

NOTEBOOK 08 — CELL 11: PASS


{'notebook': '08',
 'cell': 11,
 'status': 'PASS',
 'version': 'stage08-main-v4-complete-accounting',
 'tasks': 1700,
 'opportunities': 22238,
 'request_attempts': 1897,
 'collected_item_outcomes': 1012,
 'no_silent_population_exclusion': True}

In [13]:
# ============================================================
# NOTEBOOK 08 — CELL 12
# Collect assertion propositions and qualify executed expression reachability.
# ============================================================

from __future__ import annotations

from collections import Counter
import json
import time


require_pass(11)
ASSERTION_CLAIMS = []
TRACE_REPORTS = []
for index, task in enumerate(TASKS, 1):
    task_id = task["task_id"]
    item = next(r for r in INPUT_INVENTORY if r["task_id"] == task_id)
    if not item["extracted_sha256"]:
        continue
    if STATIC_BY_TASK[task_id]["compile_status"] != "PASS":
        continue
    source = frozen_code(task_id)
    nodes = claims.assertion_nodes(source)
    baseline = BASELINE_BY_TASK[task_id]
    reached = None
    trace_status = "NOT_EXECUTED"
    if COLLECTION_BY_TASK[task_id]["status"] == "PASS":
        probe_path = PROJECT_ROOT / "runs/main/claim_trace" / (task_id + ".json")
        if probe_path.is_file():
            probe = read_json(probe_path)
        else:
            result = worker.evaluate_isolated(
                claims.observation_wrapper(source), frozen_route(task["op_id"]),
                phase="run", timeout=35, maximum_output=200000)
            if result["status"] == "EVALUATOR_FAILURE":
                raise RuntimeError("Evaluator failed during claim observation: " + task_id)
            observed = claims.parse_observed_positions(result["stdout"])
            def request_signature(rows):
                return [(row.get("pytest_item_id"), row.get("status"), row.get("request"),
                         row.get("scenario_id"), row.get("reason_code"))
                        for row in rows]

            same_requests = request_signature(result["request_attempts"]) == request_signature(
                baseline["request_attempts"])
            same_status = (result["status"] == baseline["status"]
                           and (result["item_outcomes"] or {}) == baseline["item_outcomes"])
            valid = observed is not None and same_requests and same_status
            probe = {"task_id": task_id, "status": "TRACE_MATCH" if valid else "TRACE_UNVERIFIED",
                     "observed_positions": observed if valid else None,
                     "primary_status": baseline["status"],
                     "trace_status": result["status"],
                     "same_requests": same_requests, "same_status": same_status}
            immutable_json(probe_path, probe)
        reached = probe["observed_positions"]
        trace_status = probe["status"]
        TRACE_REPORTS.append(probe)
    requirements = [r for r in ORACLES[task["op_id"]]["requirements"]
                    if r.get("applicability") == "APPLICABLE"]
    for node in nodes:
        expression = node["expression"]
        # Give reviewers the full applicable frozen requirements for this
        # operation. A textual heuristic can suppress a relevant rule and
        # change the scientific judgment, especially for multi-status APIs.
        candidate = requirements
        claim_id = "claim_" + sha256_bytes(json_bytes(
            [task_id, node["line"], node["column"], node["ordinal"], expression]
        ))[:24]
        item_ids = claims.reached_items(node, reached) if reached is not None else []
        reachability = ("REACHED" if item_ids else
                        "NOT_REACHED" if reached is not None else "UNVERIFIED")
        ASSERTION_CLAIMS.append({
            "claim_id": claim_id, "task_id": task_id, "op_id": task["op_id"],
            "pair_id": task["pair_id"], "condition": task["condition"],
            "artifact_id": item["artifact_id"],
            "line": node["line"], "column": node["column"],
            "ordinal": node["ordinal"], "normalized_claim": expression,
            "source_span": node["source_span"], "reachability": reachability,
            "trace_status": trace_status, "reached_item_ids": item_ids,
            "requirement_candidates": [r["requirement_id"] for r in candidate],
            "observed_scenario_ids": sorted({r["scenario_id"] for r in baseline["request_attempts"]
                                             if r.get("scenario_id") and r.get("pytest_item_id") in item_ids}),
        })
    if index % 40 == 0:
        print({"claim_source_tasks_checked": index, "expected": MAIN_TASKS})

if len({r["claim_id"] for r in ASSERTION_CLAIMS}) != len(ASSERTION_CLAIMS):
    raise RuntimeError("Duplicate immutable claim identities")
write_jsonl(PROJECT_ROOT / "data/evidence/assertion_claims_internal.jsonl",
            ASSERTION_CLAIMS)
write_jsonl(PROJECT_ROOT / "data/evidence/claim_reachability_qualification.jsonl",
            TRACE_REPORTS)
REACHED_CLAIMS = [r for r in ASSERTION_CLAIMS if r["reachability"] == "REACHED"]
report_cell(12, extracted_assertion_claims=len(ASSERTION_CLAIMS),
            reached_claims=len(REACHED_CLAIMS),
            reachability_counts=dict(Counter(r["reachability"] for r in ASSERTION_CLAIMS)))

if len({row["claim_id"] for row in ASSERTION_CLAIMS}) != len(ASSERTION_CLAIMS):
    raise RuntimeError("Main claim frame contains duplicate identities.")
BASELINE_ATTEMPTS = read_jsonl(PROJECT_ROOT / "data/evidence/request_attempts.jsonl")
immutable_json(PROJECT_ROOT / "config/stage08_review_protocol.json", {
    "review_protocol_version": STAGE08_VERSION, "partition": "main", "upstream": SOURCE_IDENTITIES,
    "reviewer_1_required": len(ASSERTION_CLAIMS), "reviewer_2_required": min(len(ASSERTION_CLAIMS),300),
    "claim_frame_sha256": sha256_file(PROJECT_ROOT / "data/evidence/assertion_claims_internal.jsonl"),
})


{'claim_source_tasks_checked': 40, 'expected': 1700}
{'claim_source_tasks_checked': 80, 'expected': 1700}
{'claim_source_tasks_checked': 120, 'expected': 1700}


{'claim_source_tasks_checked': 160, 'expected': 1700}
{'claim_source_tasks_checked': 200, 'expected': 1700}


{'claim_source_tasks_checked': 240, 'expected': 1700}
{'claim_source_tasks_checked': 280, 'expected': 1700}
{'claim_source_tasks_checked': 320, 'expected': 1700}


{'claim_source_tasks_checked': 360, 'expected': 1700}
{'claim_source_tasks_checked': 400, 'expected': 1700}


{'claim_source_tasks_checked': 440, 'expected': 1700}
{'claim_source_tasks_checked': 480, 'expected': 1700}
{'claim_source_tasks_checked': 520, 'expected': 1700}


{'claim_source_tasks_checked': 560, 'expected': 1700}
{'claim_source_tasks_checked': 600, 'expected': 1700}


... (7 outputs left out)

{'claim_source_tasks_checked': 1400, 'expected': 1700}
{'claim_source_tasks_checked': 1440, 'expected': 1700}
{'claim_source_tasks_checked': 1480, 'expected': 1700}
{'claim_source_tasks_checked': 1520, 'expected': 1700}
{'claim_source_tasks_checked': 1560, 'expected': 1700}


{'claim_source_tasks_checked': 1600, 'expected': 1700}
{'claim_source_tasks_checked': 1640, 'expected': 1700}
{'claim_source_tasks_checked': 1680, 'expected': 1700}


{
  "notebook": "08",
  "cell": 12,
  "status": "PASS",
  "version": "stage08-main-v4-complete-accounting",
  "extracted_assertion_claims": 5348,
  "reached_claims": 40,
  "reachability_counts": {
    "NOT_REACHED": 2905,
    "UNVERIFIED": 2403,
    "REACHED": 40
  }
}

NOTEBOOK 08 — CELL 12: PASS


'REUSED_IDENTICAL'

In [14]:
require_pass(12)
# Use the same review key for every session of this corrected run.
import hmac
import secrets
from cryptography.hazmat.primitives.ciphers.aead import AESGCM

KEY_PATH = WORK_ROOT / "private/blinding_key.bin"
encrypted_path = PROJECT_ROOT / "private/blind_mapping.aesgcm"
key_source = os.environ.get("STAGE08_BLINDING_KEY_FILE", "").strip()
if key_source:
    immutable_write(KEY_PATH, Path(key_source).read_bytes())
if not KEY_PATH.is_file():
    if encrypted_path.exists():
        raise RuntimeError("BLINDING_BLOCK: the corrected run's private key is missing")
    immutable_write(KEY_PATH, secrets.token_bytes(32))
KEY_PATH.chmod(0o600)
BLINDING_KEY = KEY_PATH.read_bytes()
if len(BLINDING_KEY) != 32:
    raise RuntimeError("Expected a 32-byte blinding key")

def blind_identifier(prefix, identity):
    message = f"{STAGE08_VERSION}:{prefix}:{identity}".encode("utf-8")
    return prefix + "_" + hmac.new(BLINDING_KEY, message, hashlib.sha256).hexdigest()[:24]


def blind_case_context(attempt):
    return {key: attempt.get(key) for key in
            ("pytest_item_id", "item_request_ordinal", "status", "scenario_id", "response_status")}


attempts_by_task_and_item = defaultdict(list)
for attempt in BASELINE_ATTEMPTS:
    attempts_by_task_and_item[attempt["task_id"], attempt.get("pytest_item_id")].append(
        blind_case_context(attempt)
    )

BLIND_MAP = {}
BLINDED_CLAIMS = []
BLIND_TEST_CONTEXT = {}
BLIND_SPEC_CONTEXT = {}
REQUIREMENTS_BY_OPERATION = {}
for claim in ASSERTION_CLAIMS:
    op_id = claim["op_id"]
    artifact_id = claim["artifact_id"]
    blind_id = blind_identifier("bc", claim["claim_id"])
    test_context_id = blind_identifier("bt", artifact_id)
    spec_context_id = blind_identifier("bs", op_id)
    if blind_id in BLIND_MAP:
        raise RuntimeError("Blinded claim identity collision")
    BLIND_MAP[blind_id] = {"claim_id": claim["claim_id"], "task_id": claim["task_id"],
                          "op_id": op_id, "condition": claim["condition"],
                          "pair_id": claim["pair_id"]}
    if spec_context_id not in BLIND_SPEC_CONTEXT:
        document = ORACLES[op_id]["operation_document"]
        requirement_rows = ORACLES[op_id]["requirements"]
        REQUIREMENTS_BY_OPERATION[op_id] = {r["requirement_id"]: r for r in requirement_rows}
        BLIND_SPEC_CONTEXT[spec_context_id] = {
            "spec_context_id": spec_context_id, "openapi_document": document
        }
    if test_context_id not in BLIND_TEST_CONTEXT:
        source_bytes = (PROJECT_ROOT / f"extracted/{artifact_id}/test_generated.py").read_bytes()
        if sha256_bytes(source_bytes) != ARTIFACT_BY_TASK[claim["task_id"]]["extracted_hash"]:
            raise RuntimeError("Frozen test source differs from the generation ledger")
        BLIND_TEST_CONTEXT[test_context_id] = {
            "test_context_id": test_context_id, "source": source_bytes.decode("utf-8")
        }
    requirements = REQUIREMENTS_BY_OPERATION[op_id]
    evidence = []
    for requirement_id in claim["requirement_candidates"]:
        requirement = requirements[requirement_id]
        evidence.append({key: requirement.get(key) for key in
                         ("requirement_id", "kind", "expected_rule", "source_pointer", "evidence_sha256")})
    BLINDED_CLAIMS.append({
        "blind_claim_id": blind_id,
        "normalized_claim": claim["normalized_claim"], "source_span": claim["source_span"],
        "line": claim["line"], "column": claim["column"],
        "test_context_id": test_context_id, "spec_context_id": spec_context_id,
        "reachability": claim["reachability"],
        "observed_scenario_ids": claim["observed_scenario_ids"],
        "reached_case_evidence": [
            {"pytest_item_id": item_id,
             "baseline_request_attempts": attempts_by_task_and_item[claim["task_id"], item_id]}
            for item_id in claim["reached_item_ids"]
        ],
        "candidate_evidence": evidence,
    })
BLINDED_CLAIMS.sort(key=lambda row: row["blind_claim_id"])

associated_data = json_bytes(read_json(PROJECT_ROOT / "config/stage08_review_protocol.json"))
if encrypted_path.is_file():
    payload = encrypted_path.read_bytes()
    restored = AESGCM(BLINDING_KEY).decrypt(payload[:12], payload[12:], associated_data)
    if json.loads(restored) != BLIND_MAP:
        raise RuntimeError("BLINDING_BLOCK: mapping differs from the frozen review frame")
else:
    nonce = secrets.token_bytes(12)
    immutable_write(encrypted_path, nonce + AESGCM(BLINDING_KEY).encrypt(
        nonce, json_bytes(BLIND_MAP), associated_data))

OUTGOING = PROJECT_ROOT / "reviews/outgoing"
write_jsonl(OUTGOING / "context_bundle.jsonl", BLINDED_CLAIMS)
write_jsonl(OUTGOING / "test_context.jsonl",
            [BLIND_TEST_CONTEXT[k] for k in sorted(BLIND_TEST_CONTEXT)])
write_jsonl(OUTGOING / "specification_context.jsonl",
            [BLIND_SPEC_CONTEXT[k] for k in sorted(BLIND_SPEC_CONTEXT)])
CONTEXT_FILES = ("context_bundle.jsonl", "test_context.jsonl", "specification_context.jsonl")
immutable_json(OUTGOING / "blind_packet_manifest.json", {
    "status": "PASS", "review_protocol_version": STAGE08_VERSION,
    "claim_count": len(BLINDED_CLAIMS),
    "context_file_sha256": {name: sha256_file(OUTGOING / name) for name in CONTEXT_FILES},
})
report_cell(13, blinded_claims=len(BLINDED_CLAIMS),
            reachability_counts=dict(Counter(row["reachability"] for row in BLINDED_CLAIMS)))
print("Keep the corrected run's key private:", KEY_PATH)


{
  "notebook": "08",
  "cell": 13,
  "status": "PASS",
  "version": "stage08-main-v4-complete-accounting",
  "blinded_claims": 5348,
  "reachability_counts": {
    "NOT_REACHED": 2905,
    "UNVERIFIED": 2403,
    "REACHED": 40
  }
}

NOTEBOOK 08 — CELL 13: PASS
Keep the corrected run's key private: /kaggle/working/stage08_main_review/private/blinding_key.bin


In [15]:
# Notebook 08 · Cell 14 · Freeze Reviewer 2’s seeded sample

from __future__ import annotations

import json
import random
import time


require_pass(13)
REVIEW_SEED = 20261111
ALL_BLIND_IDS = sorted(row["blind_claim_id"] for row in BLINDED_CLAIMS)
if len(set(ALL_BLIND_IDS)) != len(ALL_BLIND_IDS):
    raise RuntimeError("Duplicate blinded review identity")
REVIEW2_COUNT = min(len(ALL_BLIND_IDS), 300)
REVIEW2_IDS = sorted(random.Random(REVIEW_SEED).sample(ALL_BLIND_IDS, REVIEW2_COUNT))
REVIEW2_SAMPLE = {"status": "PASS", "seed": REVIEW_SEED,
                  "population_size": len(ALL_BLIND_IDS), "sample_size": REVIEW2_COUNT,
                  "sampled_ids": REVIEW2_IDS, "sampling": "uniform_without_replacement"}
immutable_json(PROJECT_ROOT / "reviews/outgoing/reviewer_2_frozen_sample.json",
               REVIEW2_SAMPLE)
report_cell(14, total_reviewable_claims=len(ALL_BLIND_IDS),
            reviewer2_sample=REVIEW2_COUNT, sample_seed=REVIEW_SEED)


{
  "notebook": "08",
  "cell": 14,
  "status": "PASS",
  "version": "stage08-main-v4-complete-accounting",
  "total_reviewable_claims": 5348,
  "reviewer2_sample": 300,
  "sample_seed": 20261111
}

NOTEBOOK 08 — CELL 14: PASS


{'notebook': '08',
 'cell': 14,
 'status': 'PASS',
 'version': 'stage08-main-v4-complete-accounting',
 'total_reviewable_claims': 5348,
 'reviewer2_sample': 300,
 'sample_seed': 20261111}

In [16]:
# Notebook 08 · Cell 15 · Export both independent review assignments

from __future__ import annotations

import json
import os
import time
import zipfile


require_pass(14)
REVIEW_COLUMNS = ("blind_claim_id", "reviewer_id", "label", "requirement_id",
                  "evidence_pointer", "reasoning", "adjudication_status")
ALLOWED_LABELS = ("supported", "contradicted", "undocumented_or_unsupported",
                  "ambiguous_or_unverifiable")


def review_template(blind_ids, reviewer_id):
    return [{"blind_claim_id": blind_id, "reviewer_id": reviewer_id,
             "label": "", "requirement_id": "", "evidence_pointer": "",
             "reasoning": "", "adjudication_status": "NOT_REQUIRED"}
            for blind_id in blind_ids]


write_csv(PROJECT_ROOT / "reviews/outgoing/reviewer_1_template.csv",
          REVIEW_COLUMNS, review_template(ALL_BLIND_IDS, "reviewer_1"))
write_csv(PROJECT_ROOT / "reviews/outgoing/reviewer_2_template.csv",
          REVIEW_COLUMNS, review_template(REVIEW2_IDS, "reviewer_2"))

REVIEW_GATE = {"evaluation_version": STAGE08_VERSION,
               "source_identities": SOURCE_IDENTITIES,
               "status": "AWAITING_REVIEWER_1_AND_2",
               "reviewer_1_required": len(ALL_BLIND_IDS),
               "reviewer_2_required": len(REVIEW2_IDS),
               "reviewer_3_required": "ONLY_IF_DISAGREEMENTS",
               "unblinding_authorized": False,
               "completed_stage08_dataset_authorized": False}
immutable_json(PROJECT_ROOT / "reports/checkpoint_c_review_gate.json", REVIEW_GATE)

REVIEW_INSTRUCTIONS = '# Independent assertion-claim review\n\nReviewer 1 labels all assigned claims. Reviewer 2 labels only the frozen sample, independently. Reviewer 3 adjudicates only label disagreements using the two prior decisions and the blinded evidence.\n\nFor each blind_claim_id, find the matching row in context_bundle.jsonl. Read the assertion and the source location. Use test_context_id to look up the test in test_context.jsonl and spec_context_id to look up the OpenAPI specification in specification_context.jsonl. Each JSONL line contains one JSON object. Check the applicable scenario, test inputs and expected value. For reached assertions, use reached_case_evidence to join the pytest case to its recorded scenario and response. For other assertions, judge the documented contract against the scenario inferable from the test and record any uncertainty. Reachability is recorded independently from the semantic label.\n\n- supported — the applicable specification justifies the expectation.\n- contradicted — applicable evidence conflicts with the expectation.\n- undocumented_or_unsupported — the expectation is clear but the applicable specification does not establish it.\n- ambiguous_or_unverifiable — a necessary value, scenario, or rule cannot be resolved from the evidence.\n\nFill in label and reasoning for every assigned row. For supported and contradicted, copy requirement_id and source_pointer from the same candidate evidence entry, using evidence_pointer for the source pointer. For the other labels, leave both fields blank or cite a matching pair. Keep the headers, blind IDs and reviewer ID intact. Reviewer 1 and 2 keep NOT_REQUIRED. Reviewer 3 marks every final decision COMPLETED, including ambiguous decisions. Return the CSV as reviewer_1_completed.csv, reviewer_2_completed.csv or reviewer_3_completed.csv, according to your assigned role.\n'

immutable_write(OUTGOING / "REVIEW_INSTRUCTIONS.md", REVIEW_INSTRUCTIONS.encode("utf-8"))


def write_packet(filename, payloads):
    path = CHECKPOINT_ROOT / filename
    if not path.is_file():
        temporary = path.with_suffix(".tmp")
        with zipfile.ZipFile(temporary, "w", compression=zipfile.ZIP_DEFLATED) as archive:
            for name, data in payloads.items():
                archive.writestr(name, data)
        os.replace(temporary, path)
    with zipfile.ZipFile(path) as archive:
        names = archive.namelist()
        if (len(names) != len(payloads) or set(names) != set(payloads)
                or any(archive.read(name) != data for name, data in payloads.items())):
            raise RuntimeError(f"Review packet differs from the frozen files: {path}")
    return path


packet_files = (*CONTEXT_FILES, "REVIEW_INSTRUCTIONS.md", "blind_packet_manifest.json",
                "reviewer_1_template.csv", "reviewer_2_template.csv", "reviewer_2_frozen_sample.json")
packet = write_packet("stage_08_main_blind_reviewer_packets.zip",
                      {name: (OUTGOING / name).read_bytes() for name in packet_files})
report_cell(15, checkpoint_c="AWAITING_REVIEWER_1_AND_2",
            reviewer1_rows=len(ALL_BLIND_IDS), reviewer2_rows=len(REVIEW2_IDS),
            reviewer_packet=str(packet))
print("CHECKPOINT C: Pause here and obtain the two independent reviews.")
print("Share the reviewer packet:", packet)


def save_private_checkpoint(filename):
    root = WORK_ROOT / "operator_checkpoints"
    root.mkdir(parents=True, exist_ok=True)
    destination = root / filename
    if not destination.is_file():
        temporary = destination.with_suffix(".tmp")
        with zipfile.ZipFile(temporary, "w", zipfile.ZIP_DEFLATED, compresslevel=4) as archive:
            for path in sorted(PROJECT_ROOT.rglob("*")):
                if not path.is_file() or path.is_symlink() or "__pycache__" in path.parts:
                    continue
                relative = path.relative_to(PROJECT_ROOT).as_posix()
                if path.resolve() == KEY_PATH.resolve() or path.name == "blinding_key.bin":
                    raise RuntimeError("Private key must be outside the checkpoint project tree.")
                archive.write(path, "prose_operation_experiment/" + relative)
        with zipfile.ZipFile(temporary) as archive:
            if archive.testzip() is not None:
                raise RuntimeError("Operator checkpoint archive failed verification.")
        os.replace(temporary, destination)
    checksum = sha256_file(destination)
    immutable_write(root / (filename + ".sha256"), (checksum + "  " + filename + "\n").encode())
    print("Keep the operator checkpoint private:", destination)


save_private_checkpoint("stage_08_main_review_checkpoint.zip")


{
  "notebook": "08",
  "cell": 15,
  "status": "PASS",
  "version": "stage08-main-v4-complete-accounting",
  "checkpoint_c": "AWAITING_REVIEWER_1_AND_2",
  "reviewer1_rows": 5348,
  "reviewer2_rows": 300,
  "reviewer_packet": "/kaggle/working/stage08_main_review/review_packets/stage_08_main_blind_reviewer_packets.zip"
}

NOTEBOOK 08 — CELL 15: PASS
CHECKPOINT C: Pause here and obtain the two independent reviews.
Share the reviewer packet: /kaggle/working/stage08_main_review/review_packets/stage_08_main_blind_reviewer_packets.zip


Keep the operator checkpoint private: /kaggle/working/stage08_main_review/operator_checkpoints/stage_08_main_review_checkpoint.zip


In [17]:
# Notebook 08 · Cell 16 · Validate Reviewer 1’s full review

from __future__ import annotations

import csv
import json
import os


require_pass(15)


def discover_review_file(environment_name, expected_name):
    override = os.environ.get(environment_name, "").strip()
    if override:
        path = Path(override)
        if not path.is_file():
            raise RuntimeError("REVIEW_BLOCK: configured review file missing: " + override)
        return path
    matches = sorted(p for p in SOURCE_ROOT.rglob(expected_name)
                     if p.is_file() and not p.is_symlink())
    if len(matches) != 1:
        raise RuntimeError("REVIEW_BLOCK: expected one " + expected_name
                           + f" on Kaggle inputs, found {len(matches)}")
    return matches[0]


def validated_reviews(path, expected_ids, reviewer_id):
    with Path(path).open(newline="", encoding="utf-8-sig") as stream:
        reader = csv.DictReader(stream)
        if tuple(reader.fieldnames or []) != REVIEW_COLUMNS:
            raise RuntimeError("REVIEW_BLOCK: review columns differ from frozen schema")
        rows = list(reader)
    if any(None in row or None in row.values() for row in rows):
        raise RuntimeError("REVIEW_BLOCK: CSV row width differs from the frozen schema")
    if (len(rows) != len(expected_ids)
            or len({r["blind_claim_id"] for r in rows}) != len(rows)
            or {r["blind_claim_id"] for r in rows} != set(expected_ids)):
        raise RuntimeError("REVIEW_BLOCK: reviewer identities/count differ from frozen frame")
    evidence_lookup = {r["blind_claim_id"]: r for r in BLINDED_CLAIMS}
    for row in rows:
        if row["reviewer_id"] != reviewer_id or row["label"] not in ALLOWED_LABELS:
            raise RuntimeError("REVIEW_BLOCK: illegal reviewer ID or label")
        if not row["reasoning"].strip():
            raise RuntimeError("REVIEW_BLOCK: reasoning cannot be omitted")
        if row["adjudication_status"] not in ("NOT_REQUIRED", "COMPLETED"):
            raise RuntimeError("REVIEW_BLOCK: unknown adjudication status")
        allowed = {x["requirement_id"]: x["source_pointer"]
                   for x in evidence_lookup[row["blind_claim_id"]]["candidate_evidence"]}
        if row["label"] in ("supported", "contradicted"):
            if (row["requirement_id"] not in allowed
                    or row["evidence_pointer"] != allowed[row["requirement_id"]]):
                raise RuntimeError("REVIEW_BLOCK: supported/contradicted label needs "
                                   "a matching frozen requirement evidence pointer")
        else:
            if row["requirement_id"] and row["requirement_id"] not in allowed:
                raise RuntimeError("REVIEW_BLOCK: unknown requirement ID")
            if (row["requirement_id"]
                    and row["evidence_pointer"] != allowed[row["requirement_id"]]):
                raise RuntimeError("REVIEW_BLOCK: inaccurate evidence pointer")
    return sorted(rows, key=lambda r: r["blind_claim_id"])


R1_SOURCE = discover_review_file("STAGE08_REVIEWER1_FILE", "reviewer_1_completed.csv")
REVIEW1_ROWS = validated_reviews(R1_SOURCE, ALL_BLIND_IDS, "reviewer_1")
write_csv(PROJECT_ROOT / "reviews/completed/reviewer_1.csv", REVIEW_COLUMNS, REVIEW1_ROWS)
report_cell(16, reviewer_1_rows=len(REVIEW1_ROWS),
            csv_sha256=sha256_file(PROJECT_ROOT / "reviews/completed/reviewer_1.csv"))


{
  "notebook": "08",
  "cell": 16,
  "status": "PASS",
  "version": "stage08-main-v4-complete-accounting",
  "reviewer_1_rows": 5348,
  "csv_sha256": "939125f300a60f95f61f66a68d7fd0d7fb1fd224a62c71a35272ecc05bfe1a94"
}

NOTEBOOK 08 — CELL 16: PASS


{'notebook': '08',
 'cell': 16,
 'status': 'PASS',
 'version': 'stage08-main-v4-complete-accounting',
 'reviewer_1_rows': 5348,
 'csv_sha256': '939125f300a60f95f61f66a68d7fd0d7fb1fd224a62c71a35272ecc05bfe1a94'}

In [18]:
# Notebook 08 · Cell 17 · Validate Reviewer 2’s independent sample

from __future__ import annotations

import json
import time


require_pass(16)
R2_SOURCE = discover_review_file("STAGE08_REVIEWER2_FILE", "reviewer_2_completed.csv")
REVIEW2_ROWS = validated_reviews(R2_SOURCE, REVIEW2_IDS, "reviewer_2")
write_csv(PROJECT_ROOT / "reviews/completed/reviewer_2.csv", REVIEW_COLUMNS, REVIEW2_ROWS)
if {r["reviewer_id"] for r in REVIEW1_ROWS} & {
    r["reviewer_id"] for r in REVIEW2_ROWS
}:
    raise RuntimeError("REVIEW_BLOCK: independent reviewer roles must be distinct")
report_cell(17, reviewer_2_rows=len(REVIEW2_ROWS),
            reviewer_2_frozen_sample_verified=True,
            csv_sha256=sha256_file(PROJECT_ROOT / "reviews/completed/reviewer_2.csv"))


{
  "notebook": "08",
  "cell": 17,
  "status": "PASS",
  "version": "stage08-main-v4-complete-accounting",
  "reviewer_2_rows": 300,
  "reviewer_2_frozen_sample_verified": true,
  "csv_sha256": "a1f1a1afea49b5869cc5ae717f722b59387292f8bbbc48d3ce730bb6cf48451f"
}

NOTEBOOK 08 — CELL 17: PASS


{'notebook': '08',
 'cell': 17,
 'status': 'PASS',
 'version': 'stage08-main-v4-complete-accounting',
 'reviewer_2_rows': 300,
 'reviewer_2_frozen_sample_verified': True,
 'csv_sha256': 'a1f1a1afea49b5869cc5ae717f722b59387292f8bbbc48d3ce730bb6cf48451f'}

In [19]:
# Notebook 08 · Cell 18 · Export the disagreement-only adjudication packet

from __future__ import annotations

import json
import zipfile


require_pass(17)
R1_LOOKUP = {r["blind_claim_id"]: r for r in REVIEW1_ROWS}
R2_LOOKUP = {r["blind_claim_id"]: r for r in REVIEW2_ROWS}
DISAGREEMENT_IDS = sorted(
    blind_id for blind_id in REVIEW2_IDS
    if R1_LOOKUP[blind_id]["label"] != R2_LOOKUP[blind_id]["label"]
)
write_csv(PROJECT_ROOT / "reviews/outgoing/reviewer_3_template.csv",
          REVIEW_COLUMNS, review_template(DISAGREEMENT_IDS, "reviewer_3"))
immutable_json(PROJECT_ROOT / "reviews/outgoing/adjudication_manifest.json", {
    "status": "AWAITING_ADJUDICATION" if DISAGREEMENT_IDS else "NO_ADJUDICATION_NEEDED",
    "reviewer_1_sha256": sha256_file(PROJECT_ROOT / "reviews/completed/reviewer_1.csv"),
    "reviewer_2_sha256": sha256_file(PROJECT_ROOT / "reviews/completed/reviewer_2.csv"),
    "disagreement_count": len(DISAGREEMENT_IDS),
    "disagreement_ids": DISAGREEMENT_IDS,
    "unblinding_authorized": False,
})

if DISAGREEMENT_IDS:
    disagreement_set = set(DISAGREEMENT_IDS)
    contexts = [r for r in BLINDED_CLAIMS if r["blind_claim_id"] in disagreement_set]
    test_ids = sorted({r["test_context_id"] for r in contexts})
    spec_ids = sorted({r["spec_context_id"] for r in contexts})
    packet = write_packet("stage_08_main_adjudication_packet.zip", {
        "reviewer_3_template.csv": (OUTGOING / "reviewer_3_template.csv").read_bytes(),
        "REVIEW_INSTRUCTIONS.md": (OUTGOING / "REVIEW_INSTRUCTIONS.md").read_bytes(),
        "context_bundle.jsonl": b"".join(map(json_bytes, contexts)),
        "test_context.jsonl": b"".join(json_bytes(BLIND_TEST_CONTEXT[k]) for k in test_ids),
        "specification_context.jsonl": b"".join(json_bytes(BLIND_SPEC_CONTEXT[k]) for k in spec_ids),
        "prior_decisions.jsonl": b"".join(json_bytes({
            "blind_claim_id": k, "reviewer_1": R1_LOOKUP[k], "reviewer_2": R2_LOOKUP[k]
        }) for k in DISAGREEMENT_IDS),
    })
    print("Pause for one adjudication per disagreement. Share:", packet)
else:
    print("The two reviewers agree on every sampled claim. Continue to Cell 19.")

report_cell(18, reviewer_2_overlap=len(REVIEW2_IDS),
            disagreements=len(DISAGREEMENT_IDS),
            checkpoint_c=("AWAITING_ADJUDICATION" if DISAGREEMENT_IDS
                          else "NO_ADJUDICATION_NEEDED"))

if DISAGREEMENT_IDS:
    save_private_checkpoint("stage_08_main_adjudication_checkpoint.zip")


Pause for one adjudication per disagreement. Share: /kaggle/working/stage08_main_review/review_packets/stage_08_main_adjudication_packet.zip
{
  "notebook": "08",
  "cell": 18,
  "status": "PASS",
  "version": "stage08-main-v4-complete-accounting",
  "reviewer_2_overlap": 300,
  "disagreements": 59,
  "checkpoint_c": "AWAITING_ADJUDICATION"
}

NOTEBOOK 08 — CELL 18: PASS


Keep the operator checkpoint private: /kaggle/working/stage08_main_review/operator_checkpoints/stage_08_main_adjudication_checkpoint.zip


In [ ]:
# Notebook 08 · Session 2 · Final cell · package the adjudication files for download
import hashlib, os, zipfile
from pathlib import Path

WORKING = Path(os.environ.get("STAGE08_WORKING", "/kaggle/working"))
REVIEW_DIR = WORKING / "stage08_main_review"
packet = REVIEW_DIR / "review_packets/stage_08_main_adjudication_packet.zip"
checkpoint = REVIEW_DIR / "operator_checkpoints/stage_08_main_adjudication_checkpoint.zip"

def sha(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()

if not packet.is_file() and not checkpoint.is_file():
    print("NO DISAGREEMENTS: no adjudication files were created.")
    print("Do NOT stop. Continue with Cells 19 to 22 in this same session.")
else:
    for required in (packet, checkpoint):
        if not required.is_file():
            raise RuntimeError("Missing expected file, do not continue: " + str(required))
    OUT = WORKING / "session2_download"
    OUT.mkdir(exist_ok=True)
    FULL = OUT / "stage08_session2_everything.zip"
    if FULL.exists():
        FULL.unlink()
    count = 0
    with zipfile.ZipFile(FULL, "w", zipfile.ZIP_DEFLATED, compresslevel=4) as archive:
        for path in sorted(WORKING.rglob("*")):
            if (not path.is_file() or path.is_symlink() or OUT in path.parents
                    or "__pycache__" in path.parts or path.suffix == ".tmp"):
                continue
            archive.write(path, path.relative_to(WORKING).as_posix())
            count += 1
    with zipfile.ZipFile(FULL) as archive:
        assert archive.testzip() is None, "zip failed its integrity test"
        names = set(archive.namelist())
    for required in (packet, checkpoint):
        assert required.relative_to(WORKING).as_posix() in names, "missing from zip: " + str(required)
    print("Files in", FULL.name, ":", count, "| size MB:", round(FULL.stat().st_size / 1e6, 2))
    print("sha256 everything.zip        :", sha(FULL))
    print("sha256 adjudication packet   :", sha(packet))
    print("sha256 adjudication checkpoint:", sha(checkpoint))
    print()
    print("DOWNLOAD:", FULL)
    print("SEND TO REVIEWER 3 ONLY:", packet)
    print("KEEP PRIVATE: the adjudication checkpoint (inside the big zip too).")
